# 🐍 BOA Constrictor — neural lossless compression of CMS physics data (Colab)

**BOA Constrictor** is a Mamba-based lossless compressor for scientific data
([Gupta, Doglioni & Elliott 2026, *Mach. Learn.: Sci. Technol.* **7** 035014](https://doi.org/10.1088/2632-2153/ae64a9),
[repo](https://github.com/AkkiG2401/boa-constrictor)). A small neural network predicts
the probability of the *next byte* given the bytes seen so far, and a **range coder**
turns those probabilities into a bitstream whose length approaches the model's entropy
`−Σ log₂ p(byte)`. Better predictions ⟶ fewer bits ⟶ higher compression — and the
process is exactly invertible, so decompression is **bit-for-bit lossless**.

This notebook is a **self-contained** conversion of the repo's Python pipeline, wired to
the **`cms_experiment`** (real CMS jet data, `CMS_DATA_float32.bin`). It needs **no repo
checkout and no directory structure** — everything (config, data, code) lives here. Just:

> **Runtime → Change runtime type → GPU** (T4 is fine), then **Runtime → Run all**.

**Pipeline:** fetch CMS data → build model → **train** (predict next byte) → **compress** →
**decompress** → **verify lossless round-trip**.

| Component | Primary (GPU) | Fallback |
|---|---|---|
| Mamba backbone | `mamba_ssm` (fused CUDA) | `mambapy` (pure-torch, runs on GPU **or** CPU) |
| Entropy coder | `gpu_range_coder` (custom CUDA, built on the fly) | `constriction` (CPU) |

The notebook prefers the fast GPU path and **degrades gracefully** if a dependency or the
CUDA build is unavailable, so it still runs on a CPU-only runtime (just slower).


## 1 · Inline configuration

The original repo is driven by `experiments/cms_experiment/cms_experiment.yaml`. Here that
config is just a Python dict — **edit these values and re-run**. Defaults mirror the CMS
experiment, with training shortened for a quick Colab run (the paper uses 40 epochs on the
full file). Defined first so the install step knows whether to fetch `mamba_ssm`.


In [1]:
# 1. Install correct PyTorch for Colab's CUDA 12.8
!pip install torch==2.8.0 --index-url https://download.pytorch.org/whl/cu128

# 2. Uninstall any broken installs
# !pip uninstall -y mamba-ssm causal-conv1d

# 3. Install Mamba using the PyTorch we just installed
!pip install https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.5.2/causal_conv1d-1.5.2+cu12torch2.8cxx11abiTRUE-cp312-cp312-linux_x86_64.whl #same as boa repo
!pip install https://github.com/state-spaces/mamba/releases/download/v2.3.2.post1/mamba_ssm-2.3.2.post1+cu12torch2.8cxx11abiTRUE-cp312-cp312-linux_x86_64.whl --no-deps #newer version since 2.5.2 is not compatible with Colab

Looking in indexes: https://download.pytorch.org/whl/cu128
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 21.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.4/322.4 MB 62.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 889.0/889.0 MB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 MB 6.8 MB/s eta 0:00:00
  Attempting uninstall: triton
    Found existing installation: triton 3.6.0
    Uninstalling triton-3.6.0:
      Successfully uninstalled triton-3.6.0
  Attempting uninstall: nvidia-nccl-cu12
    Found existing installation: nvidia-nccl-cu12 2.28.9
    Uninstalling nvidia-nccl-cu12-2.28.9:
      Successfully uninstalled nvidia-nccl-cu12-2.28.9
  Attempting uninstall: nvidia-cudnn-cu12
    Found existing installation: nvidia-cudnn-cu12 9.19.0.56
    Uninstalling nvidia-cudnn-cu12-9.19.0.56:
      Successfully uninstalled nvidia-cudnn-cu12-9.19.0.56
  Attempting uninstall: torch
    Found existing installation: t

In [2]:
import torch

CONFIG = {
    # ---- data (real CMS jet features: rows of 24 float32 values) ----
    "data_url":   "https://raw.githubusercontent.com/AkkiG2401/boa-constrictor/main/experiments/cms_experiment/CMS_DATA_float32.bin",
    "data_mb":    None,          # MB of the 47.6 MB CMS file to use (None = full file)
    "n_features": 24,         # float32 columns per jet (pt, eta, phi, mass, energies, ...)

    # ---- model (cms_experiment.yaml) ----
    "backbone":   "mambav1",  # mamba_ssm where possible, else mambapy. Also: mamba, transformer, gru, lstm, ...
    "d_model":    64,
    "num_layers": 2,
    "vocab_size": 256,        # one symbol per byte

    # ---- dataloader ----
    "seq_len":    10000,      # bytes per training sequence
    "batch_size": 5,

    # ---- training ----
    "lr":         5e-4,
    "epochs":     8,          # cms_experiment uses 40; fewer = faster demo
    "precision":  "fp32",     # fp32 | fp16 | bf16  (mixed precision is CUDA-only)
    "splits":     (0.8, 0.1, 0.1),

    # ---- compression ----
    "chunks_count": None,     # None -> auto (~2 KB/chunk). cms_experiment uses 10000 on the full file.

    # ---- toggles ----
    "try_mamba_ssm": True,    # attempt the fused CUDA kernels (best-effort install)
    "seed": 42,
}

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Torch {torch.__version__} | device = {DEVICE}"
      + (f" ({torch.cuda.get_device_name(0)})" if DEVICE == 'cuda' else " — no GPU; the CPU path works but is slow"))


Torch 2.8.0+cu128 | device = cuda (Tesla T4)


In [3]:
!pip install minGRU-pytorch
!pip install pybind11

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.2/314.2 kB 8.4 MB/s eta 0:00:00


## 2 · Install dependencies

`constriction` (CPU range coder) and `mambapy` (pure-torch Mamba) are small, pure pip
installs. The fused **`mamba_ssm`** + **`causal-conv1d`** kernels are installed
*best-effort* on GPU — they can take a few minutes to build and may not match every Colab
CUDA/torch combo. If they fail, the notebook automatically uses `mambapy` instead, so this
cell never blocks the run. (`torch`, `numpy`, `tqdm` are preinstalled on Colab.)


In [4]:
import sys, subprocess
def pip(*args):
    print("pip", *args)
    return subprocess.call([sys.executable, "-m", "pip", "install", "-q", *args])

# Always-needed, reliable installs
pip("constriction==0.4.1", "mambapy==1.2.0", "tqdm")

# Best-effort fused Mamba kernels (GPU only). Failure is fine -> mambapy fallback.
HAVE_MAMBA_SSM = False
if DEVICE == "cuda" and CONFIG["try_mamba_ssm"]:
    try:
        import mamba_ssm  # already present?
        HAVE_MAMBA_SSM = True
    except Exception:
        try:
            import importlib, mamba_ssm  # noqa
            importlib.reload(mamba_ssm)
            HAVE_MAMBA_SSM = True
        except Exception as e:
            print(f"mamba_ssm unavailable ({e}); will use the mambapy (pure-torch) fallback.")
print("mamba_ssm available:", HAVE_MAMBA_SSM)


pip constriction==0.4.1 mambapy==1.2.0 tqdm
mamba_ssm available: True


## 3 · Fetch the CMS data

`CMS_DATA_float32.bin` is a flat array of IEEE-754 `float32`s: **N rows × 24 columns**, one
row per reconstructed jet. Columns are physics features — `pt, eta, phi, mass, jet_area`,
per-particle-type energies, and multiplicities. We stream it straight from the GitHub repo
and (by default) keep a subset for a quick demo; set `CONFIG["data_mb"] = None` for the full
47.6 MB file.


In [5]:
import os, urllib.request, numpy as np

RAW = "/content/CMS_DATA_float32.bin" if os.path.isdir("/content") else "CMS_DATA_float32.bin"
if not os.path.exists(RAW):
    print("Downloading CMS data ...")
    urllib.request.urlretrieve(CONFIG["data_url"], RAW)
full_bytes = os.path.getsize(RAW)
print(f"File on disk: {full_bytes/1e6:.1f} MB ({full_bytes//4:,} float32 values)")

# Take a byte subset (aligned to a 4-byte float boundary), then write the file we compress.
nbytes = full_bytes if CONFIG["data_mb"] is None else min(full_bytes, int(CONFIG["data_mb"] * 1_000_000))
nbytes -= nbytes % (4 * CONFIG["n_features"])              # whole jets
data = np.fromfile(RAW, dtype=np.uint8, count=nbytes).tobytes()
DATA_PATH = "/content/cms_input.bin" if os.path.isdir("/content") else "cms_input.bin"
open(DATA_PATH, "wb").write(data)

rows = len(data) // 4 // CONFIG["n_features"]
feat = np.frombuffer(data, dtype=np.float32).reshape(rows, CONFIG["n_features"])
print(f"Using {len(data)/1e6:.2f} MB = {rows:,} jets x {CONFIG['n_features']} features")
print("First jet (pt, eta, phi, mass, area, ...):", np.round(feat[0, :5], 3))


File on disk: 49.9 MB (12,480,000 float32 values)
Using 49.92 MB = 520,000 jets x 24 features
First jet (pt, eta, phi, mass, area, ...): [97.294 -1.047 -1.379 16.212  0.798]


## 4 · The BOA modules

These cells materialise the **actual** BOA Python modules into the working directory, so the
rest of the notebook can `import` them exactly as the repo does — with no directory structure
required. They are library code; feel free to collapse them. What each one provides:

- **`model.py`** — `BoaConstrictor(...)` byte-predictor factory (Mamba + other backbones),
  plus `ByteDataloader` and `make_splits`. Exposes the streaming interface the coder needs:
  `init_stream()` / `step()` / `.embedding`.
- **`gpu_range_coder.py`** — a GPU range coder that **JIT-compiles a CUDA extension with
  `nvcc` on first import** and exposes `gpu.queue.RangeCoderBatch`.
- **`codec.py`** — `compress_GPU/CPU` & `decompress_GPU/CPU`: run the model step-by-step and
  drive the range coder. Picks the GPU coder when CUDA is present, else `constriction`.
  *(Patched here so a failed CUDA build falls back to the CPU codec instead of crashing.)*
- **`boa.py`** — `BOA(...)` builds a `BoaFile` container: chunking, the on-disk `.boa`
  format (header + payload + CRC index), and `compress()` / `decompress()`.
- **`train.py`** — the training / evaluation loop (cross-entropy in *bits per byte*).
- **`online_adapt.py`** — optional test-time head adaptation (not used by default).


**`model.py`** — model factory, dataloader, splits.


In [6]:
%%writefile model.py
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import inspect as _inspect

# Optional: optimised Triton kernels (pip install flash-linear-attention)
try:
    from fla.ops.rwkv6 import chunk_rwkv6 as _fla_chunk_rwkv6
    from fla.ops.rwkv6 import fused_recurrent_rwkv6 as _fla_fused_recurrent_rwkv6
    _HAS_FLA_V6 = True
except ImportError:
    _HAS_FLA_V6 = False
try:
    from fla.ops.rwkv7 import chunk_rwkv7 as _fla_chunk_rwkv7
    from fla.ops.rwkv7 import fused_recurrent_rwkv7 as _fla_fused_recurrent_rwkv7
    _HAS_FLA_V7 = True
except ImportError:
    _HAS_FLA_V7 = False


# ======================================================================
#  Backbone blocks (LSTM / GRU / minGRU / Transformer)
# ======================================================================

AVAILABLE_BACKBONES = [
    "mamba", "mambav1", "mamba2",
    "lstm", "gru", "mingru",
    "transformer",
    "rwkv6", "rwkv7", "rwkv8", "griffin", "xlstm",
]


class _FeedForward(nn.Module):
    """Position-wise feed-forward shared across backbones."""

    def __init__(self, d_model: int, expansion: int = 4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, expansion * d_model),
            nn.GELU(),
            nn.Linear(expansion * d_model, d_model),
        )

    def forward(self, x):
        return self.net(x)


# ── LSTM ──────────────────────────────────────────────────────────

class LSTMBlock(nn.Module):
    """LSTM backbone block with pre-norm residual and feed-forward."""

    def __init__(self, d_model: int):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.lstm = nn.LSTM(d_model, d_model, num_layers=1, batch_first=True)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = _FeedForward(d_model)

    def forward(self, x, inference_params=None):
        y = self.ln1(x)
        y, _ = self.lstm(y)
        x = x + y
        y = self.ln2(x)
        y = self.ff(y)
        return x + y

    def init_cache(self, batch_size: int, device):
        d = self.lstm.hidden_size
        h0 = torch.zeros(1, batch_size, d, device=device)
        c0 = torch.zeros(1, batch_size, d, device=device)
        return (h0, c0)

    def step(self, x, cache):
        h, c = cache
        y = self.ln1(x).unsqueeze(1)
        y, (h, c) = self.lstm(y, (h, c))
        y = y.squeeze(1)
        x = x + y
        y = self.ln2(x)
        y = self.ff(y)
        return x + y, (h, c)


# ── GRU ───────────────────────────────────────────────────────────

class GRUBlock(nn.Module):
    """GRU backbone block with pre-norm residual and feed-forward.

    Uses the full ``nn.GRU`` (with hidden-to-hidden weights W_hh)
    for maximum expressivity.  Training is sequential over L, but
    cuDNN fuses the kernels so wall-clock is reasonable.
    """

    def __init__(self, d_model: int):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.gru = nn.GRU(d_model, d_model, num_layers=1, batch_first=True)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = _FeedForward(d_model)

    def forward(self, x, inference_params=None):
        y = self.ln1(x)
        y, _ = self.gru(y)
        x = x + y
        y = self.ln2(x)
        y = self.ff(y)
        return x + y

    def init_cache(self, batch_size: int, device):
        d = self.gru.hidden_size
        return torch.zeros(1, batch_size, d, device=device)

    def step(self, x, cache):
        h = cache
        y = self.ln1(x).unsqueeze(1)
        y, h = self.gru(y, h)
        y = y.squeeze(1)
        x = x + y
        y = self.ln2(x)
        y = self.ff(y)
        return x + y, h


# ── minGRU  (Feng et al., 2024 — "Were RNNs All We Needed?") ─────
#
#   z_t  = σ(W_z · x_t + b_z)             gate (input-only)
#   h̃_t = W_h · x_t + b_h                candidate (input-only)
#   h_t  = (1 − z_t) ⊙ h_{t−1} + z_t ⊙ h̃_t
#
# No hidden-to-hidden weights ⇒ the linear recurrence is amenable to
# a parallel prefix (associative) scan during training.  The sequential
# fallback here is correct and simple; swap in a CUDA scan kernel for
# O(L / P) wall-clock on very long sequences.
# ──────────────────────────────────────────────────────────────────

from minGRU_pytorch.minGRU import (
    heinsen_associative_scan_log,
    log_g as _mingru_log_g,
    g as _mingru_g,
)


class _MinGRUCell(nn.Module):
    """Minimal GRU cell — log-space parallel scan (Feng et al., 2024).

    Training uses the numerically stable log-space formulation with
    Heinsen's associative scan (``cumsum`` + ``logcumsumexp`` — two CUDA
    kernels instead of L sequential Python iterations).

    Hidden states are constrained to be positive via ``g()`` (Appendix
    B.3 of the paper).  The single-step path (``step``) uses the same
    ``g()`` so training and inference are consistent.
    """

    def __init__(self, d_model: int):
        super().__init__()
        self.linear_z = nn.Linear(d_model, d_model)
        self.linear_h = nn.Linear(d_model, d_model)
        self.d_model = d_model

    def forward(self, x, h_prev=None):
        """x: [B, L, D] → [B, L, D]  (parallel log-space scan)"""
        B, L, D = x.shape
        gate_logits = self.linear_z(x)                     # [B, L, D]
        h_candidate = self.linear_h(x)                     # [B, L, D]

        # Log-space parallel scan (Appendix B.3)
        log_coeffs = -F.softplus(gate_logits)              # log(1 − σ(z))
        log_z      = -F.softplus(-gate_logits)             # log(σ(z))
        log_values = log_z + _mingru_log_g(h_candidate)    # log(z · g(h̃))

        if h_prev is not None:
            log_values = torch.cat([h_prev.clamp(min=1e-8).log().unsqueeze(1),
                                    log_values], dim=1)
            log_coeffs = F.pad(log_coeffs, (0, 0, 1, 0))

        h = heinsen_associative_scan_log(log_coeffs, log_values)
        return h[:, -L:]

    def step(self, x, h_prev):
        """Single-step recurrence.  x, h_prev: [B, D]"""
        z = torch.sigmoid(self.linear_z(x))
        h_tilde = _mingru_g(self.linear_h(x))
        h = (1 - z) * h_prev + z * h_tilde
        return h, h


class MinGRUBlock(nn.Module):
    """minGRU backbone block with pre-norm residual and feed-forward."""

    def __init__(self, d_model: int):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.mingru = _MinGRUCell(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = _FeedForward(d_model)

    def forward(self, x, inference_params=None):
        y = self.ln1(x)
        y = self.mingru(y)
        x = x + y
        y = self.ln2(x)
        y = self.ff(y)
        return x + y

    def init_cache(self, batch_size: int, device):
        return torch.zeros(batch_size, self.mingru.d_model, device=device)

    def step(self, x, cache):
        y = self.ln1(x)
        y, cache = self.mingru.step(y, cache)
        x = x + y
        y = self.ln2(x)
        y = self.ff(y)
        return x + y, cache


# ── Transformer  (RoPE + GQA + SwiGLU + Attention-Sink Sliding-Window)
#
# Three targeted fixes over the vanilla version:
#
# 1. **No train/test mismatch** – windowed causal attention is used
#    during training too (when the sequence is longer than the window),
#    so the model never learns to rely on context it won't have during
#    streaming.  The first ``n_sink`` positions are always visible to
#    every query (attention sinks).
#
# 2. **SwiGLU FFN** – gated feed-forward (Shazeer 2020, used by LLaMA /
#    Mistral / DeepSeek) gives better parameter efficiency than the
#    plain GELU FFN.
#
# 3. **Grouped-Query Attention (GQA)** – shares KV heads across
#    multiple query heads (Ainslie et al. 2023), shrinking the KV
#    cache by ``n_heads / n_kv_heads`` and freeing capacity.
#
# Streaming: KV cache keeps the first ``n_sink`` tokens plus a sliding
# window of the most recent ``window_size`` tokens, bounding memory to
# O(n_sink + window_size) per layer — same as before but now matched to
# training.
#
# References
#   Xiao et al., "Efficient Streaming Language Models with Attention
#       Sinks", 2023.
#   Ainslie et al., "GQA: Training Generalized Multi-Query Transformer
#       Models from Multi-Head Checkpoints", 2023.
#   Shazeer, "GLU Variants Improve Transformer", 2020.
# ──────────────────────────────────────────────────────────────────

class _RMSNorm(nn.Module):
    """Root-Mean-Square Layer Normalization (Zhang & Sennrich 2019)."""

    def __init__(self, d_model: int, eps: float = 1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(d_model))
        self.eps = eps

    def forward(self, x):
        rms = x.float().pow(2).mean(-1, keepdim=True).add(self.eps).rsqrt()
        return (x.float() * rms).to(x.dtype) * self.weight


class _SwiGLU(nn.Module):
    """SwiGLU feed-forward: gate(x) * V(x) mapped back to d_model.

    Uses 8/3·d_model hidden dim (≈ 2.67x), similar param count to 4x GELU
    FFN but with gating.
    """

    def __init__(self, d_model: int):
        super().__init__()
        hidden = int(8 * d_model / 3)
        # Round to nearest multiple of 8 for tensor-core efficiency
        hidden = ((hidden + 7) // 8) * 8
        self.w_gate = nn.Linear(d_model, hidden, bias=False)
        self.w_up   = nn.Linear(d_model, hidden, bias=False)
        self.w_down = nn.Linear(hidden, d_model, bias=False)

    def forward(self, x):
        return self.w_down(F.silu(self.w_gate(x)) * self.w_up(x))


def _make_sliding_window_mask(L: int, window_size: int, n_sink: int,
                              device: torch.device) -> torch.Tensor:
    """Build a [L, L] bool attention mask for windowed + sink attention.

    Entry ``mask[i, j] = True`` means query-i **cannot** attend to key-j.
    Compatible with ``F.scaled_dot_product_attention(attn_mask=...)`` which
    treats True as "masked out" (−inf) when the mask dtype is bool.
    """
    # Start with everything masked
    mask = torch.ones(L, L, dtype=torch.bool, device=device)
    row = torch.arange(L, device=device).unsqueeze(1)
    col = torch.arange(L, device=device).unsqueeze(0)

    # (a) Causal: can only attend to past+self
    causal = col <= row
    # (b) Within sliding window
    in_window = (row - col) < window_size
    # (c) Sink positions (first n_sink tokens are always visible)
    is_sink = col < n_sink

    visible = causal & (in_window | is_sink)
    mask = ~visible  # True = masked-out for SDPA
    return mask


class TransformerBlock(nn.Module):
    """Transformer block: RoPE + GQA + SwiGLU + sliding-window attention.

    Parameters
    ----------
    d_model     : int   – hidden dimension.
    n_heads     : int   – number of **query** heads (default 4).
    n_kv_heads  : int   – number of KV heads for GQA.  Must divide n_heads
                          evenly.  Default = n_heads (= standard MHA).
                          Set to 1 for Multi-Query Attention.
    window_size : int   – sliding-window width (default 4096).
    n_sink      : int   – attention-sink positions (default 4).
    """

    def __init__(self, d_model: int, n_heads: int = 4, n_kv_heads: int = 0,
                 window_size: int = 4096, n_sink: int = 4):
        super().__init__()
        if n_kv_heads <= 0:
            n_kv_heads = n_heads          # default: standard MHA
        assert d_model % n_heads == 0, \
            f"d_model ({d_model}) must be divisible by n_heads ({n_heads})"
        assert n_heads % n_kv_heads == 0, \
            f"n_heads ({n_heads}) must be divisible by n_kv_heads ({n_kv_heads})"

        self.n_heads    = n_heads
        self.n_kv_heads = n_kv_heads
        self.n_rep      = n_heads // n_kv_heads   # GQA repeat factor
        self.d_head     = d_model // n_heads
        self.d_model    = d_model
        self.window_size = window_size
        self.n_sink      = n_sink

        self.ln1 = _RMSNorm(d_model)
        # Separate Q and KV projections for GQA
        self.q_proj  = nn.Linear(d_model, n_heads    * self.d_head, bias=False)
        self.kv_proj = nn.Linear(d_model, 2 * n_kv_heads * self.d_head, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.ln2 = _RMSNorm(d_model)
        self.ff = _SwiGLU(d_model)

        # RoPE inverse frequencies (not a trained parameter)
        inv_freq = 1.0 / (
            10000.0 ** (torch.arange(0, self.d_head, 2).float() / self.d_head)
        )
        self.register_buffer("rope_inv_freq", inv_freq, persistent=False)

    # ---- RoPE helpers ------------------------------------------------

    def _apply_rope(self, x, positions):
        """Apply rotary position embeddings.

        x         : [B, n_heads, L, d_head]
        positions : LongTensor [L]
        """
        freqs = torch.outer(positions.float(), self.rope_inv_freq)  # [L, dh/2]
        cos = freqs.cos().unsqueeze(0).unsqueeze(0)                 # [1,1,L,dh/2]
        sin = freqs.sin().unsqueeze(0).unsqueeze(0)
        x1 = x[..., : self.d_head // 2]
        x2 = x[..., self.d_head // 2 :]
        return torch.cat([x1 * cos - x2 * sin,
                          x2 * cos + x1 * sin], dim=-1)

    def _expand_kv(self, kv):
        """Repeat KV heads to match the number of query heads (GQA)."""
        if self.n_rep == 1:
            return kv
        B, n_kv, L, D = kv.shape
        return kv[:, :, None, :, :].expand(B, n_kv, self.n_rep, L, D) \
                   .reshape(B, self.n_heads, L, D)

    # ---- full-sequence forward (training) ----------------------------
    # Uses the SAME sliding-window + sink mask as inference so there is
    # no distribution shift.

    def forward(self, x, inference_params=None):
        B, L, D = x.shape
        y = self.ln1(x)

        q = self.q_proj(y).reshape(B, L, self.n_heads, self.d_head).transpose(1, 2)
        kv = self.kv_proj(y).reshape(B, L, 2, self.n_kv_heads, self.d_head)
        k, v = kv.unbind(dim=2)
        k = k.transpose(1, 2)         # [B, n_kv, L, dh]
        v = v.transpose(1, 2)

        positions = torch.arange(L, device=x.device)
        q = self._apply_rope(q, positions)
        k = self._apply_rope(k, positions)

        # Expand KV heads for GQA
        k = self._expand_kv(k)
        v = self._expand_kv(v)

        # Build sliding-window + sink mask (matches inference behaviour)
        if L <= self.window_size + self.n_sink:
            # Short sequence — plain causal is equivalent & faster
            y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        else:
            mask = _make_sliding_window_mask(L, self.window_size, self.n_sink,
                                             device=x.device)
            y = F.scaled_dot_product_attention(q, k, v, attn_mask=mask)

        y = y.transpose(1, 2).reshape(B, L, D)
        y = self.out_proj(y)
        x = x + y

        y = self.ln2(x)
        y = self.ff(y)
        return x + y

    # ---- streaming helpers -------------------------------------------

    def init_cache(self, batch_size: int, device):
        """Return empty KV cache: (k, v, step_count)."""
        k = torch.zeros(batch_size, self.n_kv_heads, 0, self.d_head, device=device)
        v = torch.zeros(batch_size, self.n_kv_heads, 0, self.d_head, device=device)
        return (k, v, 0)

    def step(self, x, cache):
        """Single-token step with attention-sink + sliding-window eviction.

        x     : [B, D]
        cache : (k_cache, v_cache, step_count)
        """
        k_cache, v_cache, step_count = cache
        B = x.shape[0]

        y = self.ln1(x).unsqueeze(1)                                # [B, 1, D]
        q = self.q_proj(y).reshape(B, 1, self.n_heads, self.d_head).transpose(1, 2)
        kv = self.kv_proj(y).reshape(B, 1, 2, self.n_kv_heads, self.d_head)
        k_new, v_new = kv.unbind(dim=2)
        k_new = k_new.transpose(1, 2)                               # [B, n_kv, 1, dh]
        v_new = v_new.transpose(1, 2)

        # RoPE at the current absolute position
        pos_t = torch.tensor([step_count], device=x.device)
        q     = self._apply_rope(q, pos_t)
        k_new = self._apply_rope(k_new, pos_t)

        # Append to cache
        k_cache = torch.cat([k_cache, k_new], dim=2)
        v_cache = torch.cat([v_cache, v_new], dim=2)
        step_count += 1

        # Evict: keep first n_sink entries + last window_size entries
        max_cache = self.n_sink + self.window_size
        if k_cache.shape[2] > max_cache:
            k_cache = torch.cat([k_cache[:, :, :self.n_sink],
                                 k_cache[:, :, -self.window_size:]], dim=2)
            v_cache = torch.cat([v_cache[:, :, :self.n_sink],
                                 v_cache[:, :, -self.window_size:]], dim=2)

        # Expand KV for GQA and attend
        k_exp = self._expand_kv(k_cache)
        v_exp = self._expand_kv(v_cache)

        # Expand Q to match — already has n_heads
        scale = self.d_head ** -0.5
        attn = (q @ k_exp.transpose(-2, -1)) * scale
        attn = F.softmax(attn, dim=-1)
        y = (attn @ v_exp)                                          # [B,nh,1,dh]
        y = y.transpose(1, 2).reshape(B, 1, self.d_model)
        y = self.out_proj(y).squeeze(1)                              # [B, D]

        x = x + y
        y = self.ln2(x)
        y = self.ff(y)
        return x + y, (k_cache, v_cache, step_count)


# ── RWKV-6  (Peng et al., 2024 — "Eagle and Finch") ──────────────
#
#   Linear-complexity RNN with data-dependent decay and matrix-valued
#   hidden state.  Parallelisable via a prefix-sum scan; sequential
#   fallback here for portability.
#
#   Time-mixing:
#     Token-shift interpolation → R, K, V, G projections
#     Data-dependent decay:  w_t = base_w + W_w(x_t)
#     Matrix state update:   S_t = diag(exp(w_t)) · S_{t-1} + k_t v_t^T
#     Output:                o_t = (S_t @ r_t) ⊙ gate_t
# ──────────────────────────────────────────────────────────────────

class _RWKV6TimeMix(nn.Module):
    """RWKV-6 time-mixing with optimised WKV computation.

    Uses FLA Triton kernels (``pip install flash-linear-attention``) when
    available on CUDA, otherwise falls back to a pure-PyTorch chunk-wise
    parallel implementation (O(C²) matmuls per chunk, O(L/C) sequential
    chunk iterations).  Both paths are numerically equivalent.
    """

    def __init__(self, d_model: int, n_heads: int = 4, chunk_size: int = 32):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.d_model = d_model
        self.chunk_size = chunk_size

        # Token-shift interpolation coefficients
        self.mix_r = nn.Parameter(torch.ones(d_model) * 0.5)
        self.mix_k = nn.Parameter(torch.ones(d_model) * 0.5)
        self.mix_v = nn.Parameter(torch.ones(d_model) * 0.5)
        self.mix_g = nn.Parameter(torch.ones(d_model) * 0.5)

        # Projections
        self.W_r = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_g = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)

        # Data-dependent decay  (official: -softplus(-x) - 0.5 ensures w < -0.5)
        self.base_decay = nn.Parameter(torch.zeros(n_heads, self.d_head) + 0.5)
        self.W_decay = nn.Linear(d_model, n_heads * self.d_head, bias=False)
        nn.init.zeros_(self.W_decay.weight)

        # Bonus (u) — per-head position-0 bias, as in the official RWKV
        self.bonus = nn.Parameter(torch.zeros(n_heads, self.d_head))

        self.ln_out = nn.GroupNorm(n_heads, d_model)

    def _token_shift(self, x, last_x):
        """Shift x by one position, prepending last_x (or zeros)."""
        if last_x is None:
            last_x = torch.zeros_like(x[:, 0])
        return torch.cat([last_x.unsqueeze(1), x[:, :-1]], dim=1)

    @staticmethod
    def _decay(base, delta):
        """-softplus(-(base + delta)) - 0.5  →  always < -0.5."""
        return -F.softplus(-(base + delta)) - 0.5

    # ── Pure-PyTorch chunk-wise fallback ──────────────────────────────

    def _chunk_wkv(self, r, k, v, w, initial_state):
        """Chunk-wise parallel WKV (pure PyTorch, works on CPU & GPU).

        Args:
            r, k, v: [B, L, H, dh]
            w:       [B, L, H, dh]  (log-decay, negative)
            initial_state: [B, H, dh, dh] or None
        Returns:
            out: [B, L, H, dh],  state: [B, H, dh, dh]
        """
        B, L, H, dh = r.shape
        C = self.chunk_size

        # Pad to multiple of C
        pad = (C - L % C) % C
        if pad > 0:
            r = F.pad(r, (0, 0, 0, 0, 0, pad))
            k = F.pad(k, (0, 0, 0, 0, 0, pad))
            v = F.pad(v, (0, 0, 0, 0, 0, pad))
            w = F.pad(w, (0, 0, 0, 0, 0, pad))
        Lp = L + pad
        nc = Lp // C

        r = r.view(B, nc, C, H, dh)
        k = k.view(B, nc, C, H, dh)
        v = v.view(B, nc, C, H, dh)
        w = w.view(B, nc, C, H, dh)

        w_cumsum = w.cumsum(dim=2)

        # Intra-chunk: causal attention with decay
        rel_log = w_cumsum.unsqueeze(3) - w_cumsum.unsqueeze(2)
        causal = torch.tril(torch.ones(C, C, device=r.device, dtype=torch.bool))
        causal = causal.view(1, 1, C, C, 1, 1)

        attn = (r.unsqueeze(3) * k.unsqueeze(2) * torch.exp(rel_log))
        attn = attn.masked_fill(~causal, 0.0).sum(dim=-1)   # [B, nc, Ct, Cs, H]
        o_intra = torch.einsum('bncsh,bnshd->bnchd',
                               attn.permute(0, 1, 2, 3, 4), v)

        # Inter-chunk: propagate recurrent state
        state = initial_state if initial_state is not None else r.new_zeros(B, H, dh, dh)
        inter_list = []
        for c_idx in range(nc):
            r_c = r[:, c_idx]
            decay_c = torch.exp(w_cumsum[:, c_idx])
            o_inter_c = torch.einsum('bhde,bche->bchd', state, r_c * decay_c)
            inter_list.append(o_inter_c)

            total_decay = torch.exp(w_cumsum[:, c_idx, -1])
            state = state * total_decay.unsqueeze(-1)
            kv_decay = torch.exp(w_cumsum[:, c_idx, -1:] - w_cumsum[:, c_idx])
            state = state + torch.einsum('bchd,bche->bhde',
                                         k[:, c_idx] * kv_decay, v[:, c_idx])

        o_inter = torch.stack(inter_list, dim=1)
        out = (o_intra + o_inter).reshape(B, Lp, H, dh)
        if pad > 0:
            out = out[:, :L]
        return out, state

    # ── Forward (selects FLA kernels or PyTorch fallback) ─────────────

    def forward(self, x, last_x=None, state=None):
        """x: [B, L, D] → ([B, L, D], last_x_new, state_new)."""
        B, L, D = x.shape
        H, dh = self.n_heads, self.d_head
        x_prev = self._token_shift(x, last_x)

        # Interpolated inputs
        r = self.W_r(x * self.mix_r + x_prev * (1 - self.mix_r)).view(B, L, H, dh)
        k = self.W_k(x * self.mix_k + x_prev * (1 - self.mix_k)).view(B, L, H, dh)
        v = self.W_v(x * self.mix_v + x_prev * (1 - self.mix_v)).view(B, L, H, dh)
        g = torch.sigmoid(self.W_g(x * self.mix_g + x_prev * (1 - self.mix_g)))

        # L2-normalize k per head (prevents state blow-up)
        k = F.normalize(k, p=2, dim=-1)

        w = self._decay(self.base_decay, self.W_decay(x).view(B, L, H, dh))

        # Dispatch to FLA Triton kernels when on CUDA, else PyTorch fallback
        if _HAS_FLA_V6 and x.is_cuda:
            fn = _fla_fused_recurrent_rwkv6 if L <= 64 else _fla_chunk_rwkv6
            out, s = fn(
                r=r, k=k, v=v, w=w,
                u=self.bonus,
                scale=1.0,
                initial_state=state,
                output_final_state=True,
            )
        else:
            out, s = self._chunk_wkv(r, k, v, w, state)

        out = out.reshape(B, L, D)
        out = self.ln_out(out.transpose(1, 2)).transpose(1, 2)
        out = out * g
        return self.W_o(out), x[:, -1], s

    def step(self, x, last_x, state):
        """Single-token step.  x, last_x: [B, D], state: [B, H, dh, dh]."""
        B = x.shape[0]
        H, dh = self.n_heads, self.d_head

        r = self.W_r(x * self.mix_r + last_x * (1 - self.mix_r)).view(B, H, dh)
        k = self.W_k(x * self.mix_k + last_x * (1 - self.mix_k)).view(B, H, dh)
        v = self.W_v(x * self.mix_v + last_x * (1 - self.mix_v)).view(B, H, dh)
        g = torch.sigmoid(self.W_g(x * self.mix_g + last_x * (1 - self.mix_g)))

        k = F.normalize(k, p=2, dim=-1)

        w = self._decay(self.base_decay, self.W_decay(x).view(B, H, dh))
        decay = torch.exp(w)

        state = state * decay.unsqueeze(-1) + torch.einsum('bhd,bhe->bhde', k, v)
        o = torch.einsum('bhde,bhd->bhe', state, r).reshape(B, self.d_model)
        o = self.ln_out(o.unsqueeze(-1)).squeeze(-1)
        o = o * g
        return self.W_o(o), x, state


class RWKV6Block(nn.Module):
    """RWKV-6 backbone block with pre-norm residual and SwiGLU FFN."""

    def __init__(self, d_model: int, n_heads: int = 4, chunk_size: int = 32):
        super().__init__()
        self.ln1 = _RMSNorm(d_model)
        self.time_mix = _RWKV6TimeMix(d_model, n_heads, chunk_size=chunk_size)
        self.ln2 = _RMSNorm(d_model)
        self.ff = _SwiGLU(d_model)

    def forward(self, x, inference_params=None):
        y = self.ln1(x)
        y, _, _ = self.time_mix(y)
        x = x + y
        y = self.ln2(x)
        y = self.ff(y)
        return x + y

    def init_cache(self, batch_size: int, device):
        tm = self.time_mix
        last_x = torch.zeros(batch_size, tm.d_model, device=device)
        state = torch.zeros(batch_size, tm.n_heads, tm.d_head, tm.d_head,
                            device=device)
        return (last_x, state)

    def step(self, x, cache):
        last_x, state = cache
        y = self.ln1(x)
        y, last_x, state = self.time_mix.step(y, last_x, state)
        x = x + y
        y = self.ln2(x)
        y = self.ff(y)
        return x + y, (last_x, state)


# ── RWKV-7 "Goose"  (Peng et al., 2025) ─────────────────────────
#
#   Extends RWKV-6 with a *non-diagonal* state transition matrix,
#   enabling cross-dimension state interactions (beyond TC⁰).
#
#   State update (per head):
#     S_t = (diag(w_t) + a_t b_t^T) · S_{t-1}  +  k_t v_t^T
#     o_t = S_t @ r_t
#
#   The rank-1 term  a_t b_t^T  lets the model perform "copy"
#   state transitions and recognize all regular languages.
#
#   Uses FLA Triton kernels on CUDA, sequential fallback on CPU.
# ──────────────────────────────────────────────────────────────────

class _RWKV7TimeMix(nn.Module):
    """RWKV-7 time-mixing with non-diagonal state transition."""

    def __init__(self, d_model: int, n_heads: int = 4, chunk_size: int = 32):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.d_model = d_model
        self.chunk_size = chunk_size

        # Token-shift interpolation
        self.mix_r = nn.Parameter(torch.ones(d_model) * 0.5)
        self.mix_k = nn.Parameter(torch.ones(d_model) * 0.5)
        self.mix_v = nn.Parameter(torch.ones(d_model) * 0.5)
        self.mix_g = nn.Parameter(torch.ones(d_model) * 0.5)
        self.mix_a = nn.Parameter(torch.ones(d_model) * 0.5)

        # Projections
        self.W_r = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_g = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)

        # Non-diagonal state transition: a, b projections (zero-init)
        self.W_a = nn.Linear(d_model, d_model, bias=False)
        self.W_b = nn.Linear(d_model, d_model, bias=False)
        nn.init.zeros_(self.W_a.weight)
        nn.init.zeros_(self.W_b.weight)

        # Data-dependent decay  (official: -softplus(-x) - 0.5)
        self.base_decay = nn.Parameter(torch.zeros(n_heads, self.d_head) + 0.5)
        self.W_decay = nn.Linear(d_model, n_heads * self.d_head, bias=False)
        nn.init.zeros_(self.W_decay.weight)

        # Bonus (position-0 bias)
        self.bonus = nn.Parameter(torch.zeros(n_heads, self.d_head))

        self.ln_out = nn.GroupNorm(n_heads, d_model)

    def _token_shift(self, x, last_x):
        if last_x is None:
            last_x = torch.zeros_like(x[:, 0])
        return torch.cat([last_x.unsqueeze(1), x[:, :-1]], dim=1)

    @staticmethod
    def _decay(base, delta):
        """-softplus(-(base + delta)) - 0.5  →  always < -0.5."""
        return -F.softplus(-(base + delta)) - 0.5

    def _chunk_sequential_wkv(self, r, k, v, w, a, b, initial_state):
        """Chunk-sequential WKV with non-diagonal transition.

        Processes chunks of size C sequentially, but vectorises
        the per-token state readout within each chunk so the
        Python loop runs L/C times instead of L times.
        """
        B, L, H, dh = r.shape
        C = self.chunk_size
        s = r.new_zeros(B, H, dh, dh) if initial_state is None else initial_state

        # Pad to multiple of C
        pad = (C - L % C) % C
        if pad > 0:
            r = F.pad(r, (0, 0, 0, 0, 0, pad))
            k = F.pad(k, (0, 0, 0, 0, 0, pad))
            v = F.pad(v, (0, 0, 0, 0, 0, pad))
            w = F.pad(w, (0, 0, 0, 0, 0, pad))
            a = F.pad(a, (0, 0, 0, 0, 0, pad))
            b = F.pad(b, (0, 0, 0, 0, 0, pad))
        Lp = L + pad
        nc = Lp // C

        all_outputs = []
        for ci in range(nc):
            sl = slice(ci * C, (ci + 1) * C)
            r_c, k_c, v_c = r[:, sl], k[:, sl], v[:, sl]
            w_c, a_c, b_c = w[:, sl], a[:, sl], b[:, sl]

            chunk_out = []
            for t in range(C):
                decay = torch.exp(w_c[:, t])
                bS = torch.einsum('bhd,bhde->bhe', b_c[:, t], s)
                ab_S = torch.einsum('bhd,bhe->bhde', a_c[:, t], bS)
                s = s * decay.unsqueeze(-1) + ab_S + \
                    torch.einsum('bhd,bhe->bhde', k_c[:, t], v_c[:, t])
                chunk_out.append(torch.einsum('bhde,bhd->bhe', s, r_c[:, t]))
            all_outputs.append(torch.stack(chunk_out, dim=1))

        out = torch.cat(all_outputs, dim=1)
        if pad > 0:
            out = out[:, :L]
        return out, s

    def forward(self, x, last_x=None, state=None):
        """x: [B, L, D] → ([B, L, D], last_x_new, state_new)."""
        B, L, D = x.shape
        H, dh = self.n_heads, self.d_head
        x_prev = self._token_shift(x, last_x)

        r = self.W_r(x * self.mix_r + x_prev * (1 - self.mix_r)).view(B, L, H, dh)
        k = self.W_k(x * self.mix_k + x_prev * (1 - self.mix_k)).view(B, L, H, dh)
        v = self.W_v(x * self.mix_v + x_prev * (1 - self.mix_v)).view(B, L, H, dh)
        g = torch.sigmoid(self.W_g(x * self.mix_g + x_prev * (1 - self.mix_g)))

        # L2-normalize k per head (prevents state blow-up)
        k = F.normalize(k, p=2, dim=-1)

        # Non-diagonal state transition vectors
        x_a = x * self.mix_a + x_prev * (1 - self.mix_a)
        a = torch.sigmoid(self.W_a(x_a)).view(B, L, H, dh)
        b = self.W_b(x_a).view(B, L, H, dh)

        w = self._decay(self.base_decay, self.W_decay(x).view(B, L, H, dh))

        # Dispatch: FLA Triton kernels on CUDA, else PyTorch fallback
        if _HAS_FLA_V7 and x.is_cuda:
            fn = _fla_fused_recurrent_rwkv7 if L <= 64 else _fla_chunk_rwkv7
            out, s = fn(
                r=r, w=w, k=k, v=v,
                a=-a, b=a * b,   # FLA convention: a→ -kappa, b→ kappa*a
                scale=1.0,
                initial_state=state,
                output_final_state=True,
            )
        else:
            if not _HAS_FLA_V7 and x.is_cuda:
                import warnings
                warnings.warn(
                    "RWKV-7 on CUDA without FLA is slow. "
                    "Install: pip install flash-linear-attention",
                    stacklevel=2,
                )
            out, s = self._chunk_sequential_wkv(r, k, v, w, a, b, state)

        out = out.reshape(B, L, D)
        out = self.ln_out(out.transpose(1, 2)).transpose(1, 2)
        out = out * g
        return self.W_o(out), x[:, -1], s

    def step(self, x, last_x, state):
        """Single-token step.  x, last_x: [B, D], state: [B, H, dh, dh]."""
        B = x.shape[0]
        H, dh = self.n_heads, self.d_head

        r = self.W_r(x * self.mix_r + last_x * (1 - self.mix_r)).view(B, H, dh)
        k = self.W_k(x * self.mix_k + last_x * (1 - self.mix_k)).view(B, H, dh)
        v = self.W_v(x * self.mix_v + last_x * (1 - self.mix_v)).view(B, H, dh)
        g = torch.sigmoid(self.W_g(x * self.mix_g + last_x * (1 - self.mix_g)))

        k = F.normalize(k, p=2, dim=-1)

        x_a = x * self.mix_a + last_x * (1 - self.mix_a)
        a = torch.sigmoid(self.W_a(x_a)).view(B, H, dh)
        b = self.W_b(x_a).view(B, H, dh)

        w = self._decay(self.base_decay, self.W_decay(x).view(B, H, dh))
        decay = torch.exp(w)

        # S = diag(decay) * S + (a b^T) @ S + k v^T
        bS = torch.einsum('bhd,bhde->bhe', b, state)
        ab_S = torch.einsum('bhd,bhe->bhde', a, bS)
        state = state * decay.unsqueeze(-1) + ab_S + \
            torch.einsum('bhd,bhe->bhde', k, v)

        o = torch.einsum('bhde,bhd->bhe', state, r).reshape(B, self.d_model)
        o = self.ln_out(o.unsqueeze(-1)).squeeze(-1)
        o = o * g
        return self.W_o(o), x, state


class RWKV7Block(nn.Module):
    """RWKV-7 backbone block with pre-norm residual and SwiGLU FFN."""

    def __init__(self, d_model: int, n_heads: int = 4, chunk_size: int = 32):
        super().__init__()
        self.ln1 = _RMSNorm(d_model)
        self.time_mix = _RWKV7TimeMix(d_model, n_heads, chunk_size=chunk_size)
        self.ln2 = _RMSNorm(d_model)
        self.ff = _SwiGLU(d_model)

    def forward(self, x, inference_params=None):
        y = self.ln1(x)
        y, _, _ = self.time_mix(y)
        x = x + y
        y = self.ln2(x)
        y = self.ff(y)
        return x + y

    def init_cache(self, batch_size: int, device):
        tm = self.time_mix
        last_x = torch.zeros(batch_size, tm.d_model, device=device)
        state = torch.zeros(batch_size, tm.n_heads, tm.d_head, tm.d_head,
                            device=device)
        return (last_x, state)

    def step(self, x, cache):
        last_x, state = cache
        y = self.ln1(x)
        y, last_x, state = self.time_mix.step(y, last_x, state)
        x = x + y
        y = self.ln2(x)
        y = self.ff(y)
        return x + y, (last_x, state)


# ── RWKV-8 "Heron" — ROSA  (Peng, 2025 — experimental) ──────────
#
#   Replaces the linear-recurrence attention with ROSA (Rapid Online
#   Suffix Automaton): a neurosymbolic mechanism that finds the
#   longest matching suffix of quantised Q in the history of K,
#   returning the V symbol after the match.
#
#   ⚠  EXPERIMENTAL — the suffix-matching core is inherently
#      sequential (not parallelisable on GPU).  Bit-packing and
#      output scatter are vectorised with PyTorch tensor ops.
#      Streaming (step) recomputes on full context.
#
#   Reference:  github.com/BlinkDL/RWKV-LM/tree/main/RWKV-v8
# ──────────────────────────────────────────────────────────────────

def _rosa_match(q_syms, k_syms, v_syms, max_ctx=0):
    """Suffix matching (reference impl from official RWKV-8).

    For each position i, find the longest suffix of q[0:i+1] that
    appears in k[0:i], and return v[match_end].

    Args:
        max_ctx: if >0, only look back at most max_ctx positions.
    """
    n = len(q_syms)
    idx = [0] * n
    ln = [0] * n
    for i in range(n):
        found = False
        max_w = min(i + 1, max_ctx) if max_ctx > 0 else i + 1
        start_j = max(0, i - max_ctx) if max_ctx > 0 else 0
        for w in range(max_w, 0, -1):
            t = q_syms[i + 1 - w: i + 1]
            for j in range(i - w, start_j - 1, -1):
                if k_syms[j: j + w] == t:
                    s = j + w
                    if s < n:
                        idx[i] = v_syms[s]
                    ln[i] = w
                    found = True
                    break
            if found:
                break
    return idx, ln


def _rosa_match_batch(q_packed, k_packed, v_packed, max_ctx=0):
    """Run _rosa_match over all groups, returning (idx, ln) tensors."""
    B, T, G = q_packed.shape
    idx_out = torch.zeros(B, T, G, dtype=torch.long)
    ln_out = torch.zeros(B, T, G, dtype=torch.long)
    for b in range(B):
        for g in range(G):
            qs = q_packed[b, :, g].tolist()
            ks = k_packed[b, :, g].tolist()
            vs = v_packed[b, :, g].tolist()
            idx, ln = _rosa_match(qs, ks, vs, max_ctx=max_ctx)
            idx_out[b, :, g] = torch.tensor(idx)
            ln_out[b, :, g] = torch.tensor(ln)
    return idx_out, ln_out


class _ROSA(nn.Module):
    """ROSA attention with vectorised bit-packing (RWKV-8).

    Args:
        max_ctx: maximum lookback window for suffix matching.
                 0 = unlimited (slow for long sequences).
    """

    def __init__(self, d_model: int, bits: int = 4, max_ctx: int = 512):
        super().__init__()
        assert d_model % bits == 0
        self.bits = bits
        self.n_groups = d_model // bits
        self.max_ctx = max_ctx
        self.emb = nn.Parameter(torch.ones(1, 1, d_model))
        # Pre-compute bit-shift powers
        self.register_buffer('_powers', 1 << torch.arange(bits))

    def _pack_bits(self, x):
        """Quantise & pack: [B, T, D] → [B, T, G] int symbols."""
        B, T, _ = x.shape
        xb = (x > 0).long()                              # [B, T, D]
        xb = xb.view(B, T, self.n_groups, self.bits)     # [B, T, G, bits]
        return (xb * self._powers.to(x.device)).sum(-1)   # [B, T, G]

    def _unpack_to_sign(self, syms, matched):
        """Unpack symbols to signed embedding output.

        Args:
            syms:    [B, T, G] int symbols (matched v values)
            matched: [B, T, G] bool (whether a match was found)
        Returns: [B, T, D] float
        """
        B, T, G = syms.shape
        device = self.emb.device
        syms = syms.to(device)
        matched = matched.to(device)

        # Unpack each symbol into bits: [B, T, G, bits]
        bits_expanded = ((syms.unsqueeze(-1) >> self._powers.to(device)) & 1).float()
        signs = bits_expanded * 2 - 1   # 0 → -1, 1 → +1
        # [B, T, G, bits] → [B, T, D]
        signs = signs.view(B, T, -1)
        # Apply embedding magnitude
        out = signs * self.emb  # broadcast [1,1,D]
        # Zero out unmatched positions: matched [B,T,G] → [B,T,D]
        mask = matched.unsqueeze(-1).expand(B, T, G, self.bits).reshape(B, T, -1)
        return out * mask.float()

    def forward(self, q, k, v):
        """q, k, v: [B, T, D] → [B, T, D]."""
        device = q.device
        # Vectorised bit-packing (on GPU if available)
        q_packed = self._pack_bits(q).cpu()
        k_packed = self._pack_bits(k).cpu()
        v_packed = self._pack_bits(v).cpu()

        # Suffix matching (CPU — inherently sequential)
        idx, ln = _rosa_match_batch(q_packed, k_packed, v_packed,
                                    max_ctx=self.max_ctx)

        # Vectorised output construction (back on GPU)
        return self._unpack_to_sign(idx, ln > 0)


class _RWKV8ROSA(nn.Module):
    """RWKV-8 ROSA time-mixing layer."""

    def __init__(self, d_model: int, bits: int = 4, max_ctx: int = 512):
        super().__init__()
        self.d_model = d_model
        self.time_shift = nn.ZeroPad2d((0, 0, 1, -1))
        self.x_q = nn.Parameter(torch.zeros(1, 1, d_model))
        self.x_k = nn.Parameter(torch.zeros(1, 1, d_model))
        self.x_v = nn.Parameter(torch.zeros(1, 1, d_model))
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.rosa = _ROSA(d_model, bits=bits, max_ctx=max_ctx)
        self.o_proj = nn.Linear(d_model, d_model)

    def forward(self, x):
        """x: [B, T, D] → [B, T, D]."""
        xx = self.time_shift(x) - x
        q = self.q_proj(x + xx * self.x_q)
        k = self.k_proj(x + xx * self.x_k)
        v = self.v_proj(x + xx * self.x_v)
        return self.o_proj(self.rosa(q, k, v))


class RWKV8Block(nn.Module):
    """RWKV-8 ROSA block (experimental).

    Args:
        max_ctx: suffix-matching lookback window (default 512).
                 Limits ROSA to O(max_ctx²) per token instead of O(T²).
    """

    def __init__(self, d_model: int, bits: int = 4, max_ctx: int = 512):
        super().__init__()
        self.d_model = d_model
        self.ln1 = nn.LayerNorm(d_model)
        self.rosa_mix = _RWKV8ROSA(d_model, bits=bits, max_ctx=max_ctx)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = _SwiGLU(d_model)

    def forward(self, x, inference_params=None):
        x = x + self.rosa_mix(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x

    def init_cache(self, batch_size: int, device):
        # Cache stores the full hidden-state history for recomputation
        return {'history': torch.zeros(batch_size, 0, self.d_model, device=device)}

    def step(self, x, cache):
        """Single-token step — appends to history and recomputes.

        ⚠  O(n²) per step where n is context length so far.
        """
        history = cache['history']
        # x: [B, D] → [B, 1, D]
        x_seq = torch.cat([history, x.unsqueeze(1)], dim=1)
        # Full forward on accumulated context
        out = self.forward(x_seq)
        # Return only the last token's output
        return out[:, -1], {'history': x_seq}


# ── Griffin  (De et al., 2024 — "Griffin / RecurrentGemma") ──────
#
#   Hybrid architecture: Real-Gated Linear Recurrent Unit (RG-LRU)
#   for long-range memory + local sliding-window causal attention
#   for fine-grained byte patterns.  Each block has three residual
#   sub-layers:
#     1. RG-LRU  (linear recurrence, O(1) per step)
#     2. Local causal attention (small window, bounded KV cache)
#     3. SwiGLU FFN
#
#   References
#     De et al., "Griffin: Mixing Gated Linear Recurrences with
#         Local Attention for Efficient Language Models", 2024.
# ──────────────────────────────────────────────────────────────────

class _RGLRU(nn.Module):
    """Real-Gated Linear Recurrent Unit — diagonal SSM with input-
    dependent gating.

    Recurrence:
        λ_t = exp(−softplus(ν) · σ(a(x_t)))       per-dim decay
        h_t = λ_t ⊙ h_{t-1} + (1 − λ_t) ⊙ gate(x_t) ⊙ proj(x_t)
    """

    def __init__(self, d_model: int):
        super().__init__()
        self.d_model = d_model
        self.input_proj = nn.Linear(d_model, d_model)
        self.gate_proj = nn.Linear(d_model, d_model)
        self.recurrence_gate = nn.Linear(d_model, d_model)
        self.log_base_decay = nn.Parameter(torch.ones(d_model) * -3.0)

    def _decay(self, x):
        a = torch.sigmoid(self.recurrence_gate(x))
        return torch.exp(-F.softplus(self.log_base_decay) * a)

    def forward(self, x):
        """x: [B, L, D] → [B, L, D]."""
        B, L, D = x.shape
        inp = self.input_proj(x) * torch.sigmoid(self.gate_proj(x))
        lam = self._decay(x)          # [B, L, D]
        h = x.new_zeros(B, D)
        outputs = []
        for t in range(L):
            h = lam[:, t] * h + (1 - lam[:, t]) * inp[:, t]
            outputs.append(h)
        return torch.stack(outputs, dim=1)

    def step(self, x, h):
        """x, h: [B, D] → ([B, D], h_new)."""
        inp = self.input_proj(x) * torch.sigmoid(self.gate_proj(x))
        lam = self._decay(x.unsqueeze(1)).squeeze(1)
        h = lam * h + (1 - lam) * inp
        return h, h


class GriffinBlock(nn.Module):
    """Griffin-style hybrid: RG-LRU + local attention + SwiGLU FFN.

    Parameters
    ----------
    d_model      : int  – hidden dimension.
    n_heads      : int  – number of attention heads for local attention.
    local_window : int  – sliding-window width for local attention.
    """

    def __init__(self, d_model: int, n_heads: int = 4, local_window: int = 128):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.d_model = d_model
        self.local_window = local_window

        # Sub-layer 1: RG-LRU
        self.ln1 = _RMSNorm(d_model)
        self.rg_lru = _RGLRU(d_model)

        # Sub-layer 2: local causal attention
        self.ln2 = _RMSNorm(d_model)
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

        # Sub-layer 3: FFN
        self.ln3 = _RMSNorm(d_model)
        self.ff = _SwiGLU(d_model)

    def _local_attention(self, x):
        B, L, D = x.shape
        q = self.q_proj(x).view(B, L, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, L, self.n_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, L, self.n_heads, self.d_head).transpose(1, 2)

        if L <= self.local_window:
            y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        else:
            mask = _make_sliding_window_mask(L, self.local_window, 0, x.device)
            y = F.scaled_dot_product_attention(q, k, v, attn_mask=mask)
        return self.out_proj(y.transpose(1, 2).reshape(B, L, D))

    def forward(self, x, inference_params=None):
        # Recurrence
        y = self.ln1(x)
        y = self.rg_lru(y)
        x = x + y
        # Local attention
        y = self.ln2(x)
        y = self._local_attention(y)
        x = x + y
        # FFN
        y = self.ln3(x)
        y = self.ff(y)
        return x + y

    def init_cache(self, batch_size: int, device):
        h_rec = torch.zeros(batch_size, self.d_model, device=device)
        k_cache = torch.zeros(batch_size, self.n_heads, 0, self.d_head,
                              device=device)
        v_cache = torch.zeros(batch_size, self.n_heads, 0, self.d_head,
                              device=device)
        return (h_rec, k_cache, v_cache)

    def step(self, x, cache):
        h_rec, k_cache, v_cache = cache
        B = x.shape[0]

        # Recurrence step
        y = self.ln1(x)
        y, h_rec = self.rg_lru.step(y, h_rec)
        x = x + y

        # Local attention step (rolling KV cache)
        y = self.ln2(x).unsqueeze(1)                                 # [B,1,D]
        q = self.q_proj(y).view(B, 1, self.n_heads, self.d_head).transpose(1, 2)
        k_new = self.k_proj(y).view(B, 1, self.n_heads, self.d_head).transpose(1, 2)
        v_new = self.v_proj(y).view(B, 1, self.n_heads, self.d_head).transpose(1, 2)

        k_cache = torch.cat([k_cache, k_new], dim=2)
        v_cache = torch.cat([v_cache, v_new], dim=2)
        if k_cache.shape[2] > self.local_window:
            k_cache = k_cache[:, :, -self.local_window:]
            v_cache = v_cache[:, :, -self.local_window:]

        scale = self.d_head ** -0.5
        attn = (q @ k_cache.transpose(-2, -1)) * scale
        attn = F.softmax(attn, dim=-1)
        y = (attn @ v_cache).transpose(1, 2).reshape(B, 1, self.d_model)
        y = self.out_proj(y).squeeze(1)
        x = x + y

        # FFN
        y = self.ln3(x)
        y = self.ff(y)
        return x + y, (h_rec, k_cache, v_cache)


# ── xLSTM  (Beck et al., 2024 — "xLSTM: Extended LSTM") ─────────
#
#   mLSTM variant with:
#     • Exponential gating (input gate can exceed 1 for amplification)
#     • Matrix-valued cell state  C ∈ R^{d_head × d_head}
#     • Covariance normaliser for numerical stability
#     • Log-space stabilisation of gates (max-trick)
#
#   Recurrence (per head, in log-space-stabilised form):
#     m_t = max(log_f_t + m_{t-1},  log_i_t)
#     f'  = exp(log_f + m_{t-1} − m_t)
#     i'  = exp(log_i − m_t)
#     C_t = f' · C_{t-1}  +  i' · (v_t ⊗ k_t)
#     n_t = f' · n_{t-1}  +  i' · k_t
#     h_t = C_t @ q_t  /  max(|n_t · q_t|, 1)
#
#   References
#     Beck et al., "xLSTM: Extended Long Short-Term Memory", 2024.
# ──────────────────────────────────────────────────────────────────

class _mLSTMCell(nn.Module):
    """mLSTM cell — matrix memory with exponential gating."""

    def __init__(self, d_model: int, n_heads: int = 4):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.d_model = d_model

        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)

        # Scalar gates per head
        self.W_i = nn.Linear(d_model, n_heads)   # input gate  (exp)
        self.W_f = nn.Linear(d_model, n_heads)   # forget gate (sigmoid)

        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.group_norm = nn.GroupNorm(n_heads, d_model)

    def forward(self, x, state=None):
        """x: [B, L, D] → ([B, L, D], new_state).

        state = (C, n, m)  with shapes
            C: [B, H, dh, dh]   matrix cell
            n: [B, H, dh]       normaliser
            m: [B, H]           log-space stabiliser
        """
        B, L, D = x.shape
        H, dh = self.n_heads, self.d_head

        q = self.W_q(x).view(B, L, H, dh)
        k = self.W_k(x).view(B, L, H, dh) * (dh ** -0.5)
        v = self.W_v(x).view(B, L, H, dh)

        log_f = -F.softplus(-self.W_f(x))         # log(σ(·)), always ≤ 0
        log_i = self.W_i(x)                        # can be > 0

        if state is None:
            C = x.new_zeros(B, H, dh, dh)
            n = x.new_zeros(B, H, dh)
            m = x.new_zeros(B, H)
        else:
            C, n, m = state

        outputs = []
        for t in range(L):
            lf = log_f[:, t]                       # [B, H]
            li = log_i[:, t]
            m_new = torch.max(lf + m, li)
            f_prime = torch.exp(lf + m - m_new)    # [B, H]
            i_prime = torch.exp(li - m_new)
            m = m_new

            k_t, v_t, q_t = k[:, t], v[:, t], q[:, t]

            C = f_prime.unsqueeze(-1).unsqueeze(-1) * C + \
                i_prime.unsqueeze(-1).unsqueeze(-1) * \
                torch.einsum('bhd,bhe->bhde', k_t, v_t)
            n = f_prime.unsqueeze(-1) * n + i_prime.unsqueeze(-1) * k_t

            h_t = torch.einsum('bhde,bhd->bhe', C, q_t)
            denom = torch.einsum('bhd,bhd->bh', n, q_t) \
                         .unsqueeze(-1).abs().clamp(min=1.0)
            outputs.append(h_t / denom)

        out = torch.stack(outputs, dim=1).reshape(B, L, D)
        out = self.group_norm(out.transpose(1, 2)).transpose(1, 2)
        return self.out_proj(out), (C, n, m)

    def step(self, x, state):
        """Single-token step.  x: [B, D], state: (C, n, m)."""
        B = x.shape[0]
        H, dh = self.n_heads, self.d_head
        C, n, m = state

        q = self.W_q(x).view(B, H, dh)
        k = self.W_k(x).view(B, H, dh) * (dh ** -0.5)
        v = self.W_v(x).view(B, H, dh)

        lf = -F.softplus(-self.W_f(x)).squeeze(-1) if self.n_heads == 1 \
             else -F.softplus(-self.W_f(x))         # [B, H]
        li = self.W_i(x) if self.n_heads > 1 else self.W_i(x)

        m_new = torch.max(lf + m, li)
        f_prime = torch.exp(lf + m - m_new)
        i_prime = torch.exp(li - m_new)
        m = m_new

        C = f_prime.unsqueeze(-1).unsqueeze(-1) * C + \
            i_prime.unsqueeze(-1).unsqueeze(-1) * \
            torch.einsum('bhd,bhe->bhde', k, v)
        n = f_prime.unsqueeze(-1) * n + i_prime.unsqueeze(-1) * k

        h = torch.einsum('bhde,bhd->bhe', C, q)
        denom = torch.einsum('bhd,bhd->bh', n, q) \
                     .unsqueeze(-1).abs().clamp(min=1.0)
        h = (h / denom).reshape(B, self.d_model)
        h = self.group_norm(h.unsqueeze(-1)).squeeze(-1)
        return self.out_proj(h), (C, n, m)


class xLSTMBlock(nn.Module):
    """xLSTM (mLSTM) backbone block with pre-norm residual and SwiGLU FFN."""

    def __init__(self, d_model: int, n_heads: int = 4):
        super().__init__()
        self.ln1 = _RMSNorm(d_model)
        self.mlstm = _mLSTMCell(d_model, n_heads)
        self.ln2 = _RMSNorm(d_model)
        self.ff = _SwiGLU(d_model)

    def forward(self, x, inference_params=None):
        y = self.ln1(x)
        y, _ = self.mlstm(y)
        x = x + y
        y = self.ln2(x)
        y = self.ff(y)
        return x + y

    def init_cache(self, batch_size: int, device):
        H = self.mlstm.n_heads
        dh = self.mlstm.d_head
        C = torch.zeros(batch_size, H, dh, dh, device=device)
        n = torch.zeros(batch_size, H, dh, device=device)
        m = torch.zeros(batch_size, H, device=device)
        return (C, n, m)

    def step(self, x, cache):
        y = self.ln1(x)
        y, cache = self.mlstm.step(y, cache)
        x = x + y
        y = self.ln2(x)
        y = self.ff(y)
        return x + y, cache


# ======================================================================
#  Model factory
# ======================================================================

def BoaConstrictor(d_model=256, num_layers=4, vocab_size=256, device="cuda",
                   backbone="mamba", **backbone_kwargs):
    """Construct a BoaBytePredictor with the specified backbone.

    Parameters
    ----------
    backbone : str
        One of: mamba, mambav1, mamba2, lstm, gru, mingru, transformer,
        rwkv6, griffin, xlstm  (default: mamba).
    **backbone_kwargs
        Extra keyword arguments forwarded to the backbone block constructor.
        For ``transformer``: n_heads, n_kv_heads, window_size, n_sink.
        For ``rwkv6``: n_heads.
        For ``griffin``: n_heads, local_window.
        For ``xlstm``: n_heads.
    """
    backbone = backbone.lower()
    if backbone not in AVAILABLE_BACKBONES:
        raise ValueError(
            f"Unknown backbone '{backbone}'. Choose from {AVAILABLE_BACKBONES}"
        )

    # ── Non-Mamba backbones ───────────────────────────────────────
    if backbone not in ("mamba", "mambav1", "mamba2"):
        _BLOCK_MAP = {
            "lstm":        LSTMBlock,
            "gru":         GRUBlock,
            "mingru":      MinGRUBlock,
            "transformer": TransformerBlock,
            "rwkv6":       RWKV6Block,
            "rwkv7":       RWKV7Block,
            "rwkv8":       RWKV8Block,
            "griffin":      GriffinBlock,
            "xlstm":       xLSTMBlock,
        }
        BlockCls = _BLOCK_MAP[backbone]

        # Only forward kwargs the block actually accepts
        _valid = (
            set(_inspect.signature(BlockCls.__init__).parameters) - {"self", "d_model"}
        )
        _bk = {k: v for k, v in backbone_kwargs.items() if k in _valid}

        class BoaBytePredictor(nn.Module):
            """Byte predictor with a non-Mamba backbone."""

            def __init__(self, d_model, num_layers, vocab_size):
                super().__init__()
                self.embedding = nn.Embedding(vocab_size, d_model)
                self.blocks = nn.ModuleList(
                    [BlockCls(d_model, **_bk) for _ in range(num_layers)]
                )
                self.final_norm = _RMSNorm(d_model)
                self.head = nn.Sequential(
                    nn.Linear(d_model, d_model),
                    nn.SiLU(),
                    nn.Linear(d_model, vocab_size),
                )
                self._backbone_name = backbone

            def forward(self, x, inference_params=None):
                h = self.embedding(x)                      # [B, L, D]
                for blk in self.blocks:
                    h = blk(h, inference_params=inference_params)
                h = self.final_norm(h)
                return self.head(h)                        # [B, L, V]

            @torch.inference_mode()
            def init_stream(self, max_len: int, batch_size: int = 1,
                            device=None, dtype=None):
                return [blk.init_cache(batch_size, device)
                        for blk in self.blocks]

            @torch.inference_mode()
            def step(self, byte_t: torch.LongTensor, caches) -> torch.Tensor:
                h = self.embedding(byte_t)                 # [B, D]
                for i, blk in enumerate(self.blocks):
                    h, caches[i] = blk.step(h, caches[i])
                h = self.final_norm(h)
                return self.head(h)                        # [B, V]

        return BoaBytePredictor(
            d_model=d_model, num_layers=num_layers, vocab_size=vocab_size,
        )

    # ── Mamba backbones ──────────────────────────────────────────
    IS_CUDA = torch.cuda.is_available() and device == "cuda"

    if IS_CUDA:
        device = "cuda"
        from mamba_ssm import Mamba
        from mamba_ssm.utils.generation import InferenceParams
    else:
        device = "cpu"
        from mambapy.mamba import MambaBlock as MambaCPU, MambaConfig

    def tag_mamba_layers_with_ids(model):
        """Give each Mamba layer a unique .layer_idx (0..N-1) for streaming cache."""
        i = 0
        for m in model.modules():
            if IS_CUDA:
                if isinstance(m, Mamba):
                    setattr(m, "layer_idx", i)
                    i += 1
            else:
                if isinstance(m, MambaCPU):
                    setattr(m, "layer_idx", i)
                    i += 1

    def bump_offset(inf, k: int = 1):
        # Most builds use seqlen_offset
        if hasattr(inf, "seqlen_offset"):
            inf.seqlen_offset += k
        elif hasattr(inf, "sequence_length_offset"):
            setattr(inf, "sequence_length_offset", getattr(inf, "sequence_length_offset") + k)
        else:
            # set a best-effort attribute for obscure builds
            setattr(inf, "seqlen_offset", getattr(inf, "seqlen_offset", 0) + k)

    # ── mambav1: original architecture from V1.1.0 ──────────────
    if backbone == "mambav1":
        class MambaBlockV1(nn.Module):
            def __init__(self, d_model: int):
                super().__init__()
                self.ln1 = nn.LayerNorm(d_model)
                if IS_CUDA:
                    self.mamba = Mamba(d_model=d_model)
                else:
                    config = MambaConfig(d_model=d_model, n_layers=0, use_cuda=False)
                    self.mamba = MambaCPU(config)
                self.ln2 = nn.LayerNorm(d_model)
                self.ff = nn.Sequential(
                    nn.Linear(d_model, 4 * d_model),
                    nn.GELU(),
                    nn.Linear(4 * d_model, d_model),
                )
            def forward(self, x, inference_params=None):
                y = self.ln1(x)
                if IS_CUDA:
                    y = self.mamba(y, inference_params=inference_params)
                else:
                    y = self.mamba(y)
                y = self.ln2(y)
                y = self.ff(y)
                return x + y

            if not IS_CUDA:
                def init_cache(self, batch_size: int, device):
                    d_inner = self.mamba.config.d_inner
                    d_conv = self.mamba.config.d_conv
                    inputs = torch.zeros(batch_size, d_inner, d_conv - 1, device=device)
                    return (None, inputs)

                def step(self, x, cache):
                    y = self.ln1(x)
                    y, cache = self.mamba.step(y, cache)
                    y = self.ln2(y)
                    y = self.ff(y)
                    return x + y, cache

        class BoaBytePredictorV1(nn.Module):
            """ Original Mamba byte predictor (V1.1.0). """
            def __init__(self, d_model=256, num_layers=4, vocab_size=256):
                super().__init__()
                self.embedding = nn.Embedding(vocab_size, d_model)
                self.blocks = nn.ModuleList([MambaBlockV1(d_model) for _ in range(num_layers)])
                self.head = nn.Sequential(
                    nn.Linear(d_model, d_model),
                    nn.ReLU(),
                    nn.Linear(d_model, vocab_size)
                )
                self._backbone_name = "mambav1"

            def forward(self, x, inference_params=None):
                h = self.embedding(x)
                for blk in self.blocks:
                    h = blk(h, inference_params=inference_params)
                return self.head(h)

            if IS_CUDA:
                @torch.inference_mode()
                def init_stream(self, max_len: int, batch_size: int = 1, device=None, dtype=None):
                    return InferenceParams(max_batch_size=batch_size, max_seqlen=max_len)

                @torch.inference_mode()
                def step(self, byte_t: torch.LongTensor, inf) -> torch.Tensor:
                    x = self.embedding(byte_t).unsqueeze(1)
                    h = x
                    for blk in self.blocks:
                        h = blk(h, inference_params=inf)
                    logits_next = self.head(h).squeeze(1)
                    bump_offset(inf, 1)
                    return logits_next
            else:
                @torch.inference_mode()
                def init_stream(self, max_len: int, batch_size: int = 1, device=None, dtype=None):
                    return [blk.init_cache(batch_size, device) for blk in self.blocks]

                @torch.inference_mode()
                def step(self, byte_t: torch.LongTensor, caches) -> torch.Tensor:
                    h = self.embedding(byte_t)
                    for i, blk in enumerate(self.blocks):
                        h, caches[i] = blk.step(h, caches[i])
                    logits_next = self.head(h)
                    return logits_next

        model = BoaBytePredictorV1(d_model=d_model, num_layers=num_layers, vocab_size=vocab_size)
        tag_mamba_layers_with_ids(model)
        return model

    # ── mamba2: Structured State Space Duality (Dao & Gu, 2024) ──
    if backbone == "mamba2":
        if not IS_CUDA:
            raise RuntimeError(
                "Mamba2 requires CUDA (mamba_ssm.modules.mamba2). "
                "Use backbone='mamba' for CPU fallback."
            )
        from mamba_ssm.modules.mamba2 import Mamba2

        class Mamba2Block(nn.Module):
            def __init__(self, d_model: int):
                super().__init__()
                self.ln1 = _RMSNorm(d_model)
                self.mamba2 = Mamba2(d_model=d_model)
                self.ln2 = _RMSNorm(d_model)
                self.ff = _SwiGLU(d_model)

            def forward(self, x, inference_params=None):
                y = self.ln1(x)
                y = self.mamba2(y, inference_params=inference_params)
                x = x + y
                y = self.ln2(x)
                y = self.ff(y)
                return x + y

        class BoaBytePredictorM2(nn.Module):
            """Mamba-2 byte predictor (SSD kernel, CUDA only)."""
            def __init__(self, d_model=256, num_layers=4, vocab_size=256):
                super().__init__()
                self.embedding = nn.Embedding(vocab_size, d_model)
                self.blocks = nn.ModuleList(
                    [Mamba2Block(d_model) for _ in range(num_layers)]
                )
                self.final_norm = _RMSNorm(d_model)
                self.head = nn.Sequential(
                    nn.Linear(d_model, d_model),
                    nn.SiLU(),
                    nn.Linear(d_model, vocab_size),
                )
                self._backbone_name = "mamba2"

            def forward(self, x, inference_params=None):
                h = self.embedding(x)
                for blk in self.blocks:
                    h = blk(h, inference_params=inference_params)
                h = self.final_norm(h)
                return self.head(h)

            @torch.inference_mode()
            def init_stream(self, max_len: int, batch_size: int = 1,
                            device=None, dtype=None):
                return InferenceParams(max_batch_size=batch_size,
                                       max_seqlen=max_len)

            @torch.inference_mode()
            def step(self, byte_t: torch.LongTensor, inf) -> torch.Tensor:
                h = self.embedding(byte_t).unsqueeze(1)
                for blk in self.blocks:
                    h = blk(h, inference_params=inf)
                h = self.final_norm(h).squeeze(1)
                logits_next = self.head(h)
                bump_offset(inf, 1)
                return logits_next

        model = BoaBytePredictorM2(d_model=d_model, num_layers=num_layers,
                                    vocab_size=vocab_size)
        tag_mamba_layers_with_ids(model)
        return model

    # ── mamba (improved): dual residual, RMSNorm, SwiGLU ────────
    class MambaBlock(nn.Module):
        def __init__(self, d_model: int):
            super().__init__()
            self.ln1 = _RMSNorm(d_model)
            if IS_CUDA:
                self.mamba = Mamba(d_model=d_model)
            else:
                config = MambaConfig(d_model=d_model, n_layers=0, use_cuda=False)
                self.mamba = MambaCPU(config)
            self.ln2 = _RMSNorm(d_model)
            self.ff = _SwiGLU(d_model)

        def forward(self, x, inference_params=None):
            # Two independent pre-norm residual streams — much better
            # gradient flow than the old single-residual path.
            y = self.ln1(x)
            if IS_CUDA:
                y = self.mamba(y, inference_params=inference_params)
            else:
                y = self.mamba(y)
            x = x + y                 # residual 1: Mamba
            y = self.ln2(x)
            y = self.ff(y)
            return x + y              # residual 2: FFN

        if not IS_CUDA:
            def init_cache(self, batch_size: int, device):
                # cache for mambapy.MambaBlock.step: (h, inputs)
                d_inner = self.mamba.config.d_inner
                d_conv = self.mamba.config.d_conv
                inputs = torch.zeros(batch_size, d_inner, d_conv - 1, device=device)
                return (None, inputs)

            def step(self, x, cache):
                # x: [B, D] -> [B, D], cache passthrough
                y = self.ln1(x)
                y, cache = self.mamba.step(y, cache)
                x = x + y             # residual 1
                y = self.ln2(x)
                y = self.ff(y)
                return x + y, cache   # residual 2

    class BoaBytePredictor(nn.Module):
        """ Mamba byte predictor.

        Improvements over the original:
        • Dual pre-norm residual streams (Mamba + FFN each have their own)
        • RMSNorm instead of LayerNorm (faster, works better with gating)
        • SwiGLU FFN instead of GELU (better param efficiency)
        • Final RMSNorm before output head (stabilises training)
        """
        def __init__(self, d_model=256, num_layers=4, vocab_size=256):
            super().__init__()
            self.embedding = nn.Embedding(vocab_size, d_model)
            self.blocks = nn.ModuleList([MambaBlock(d_model) for _ in range(num_layers)])
            self.final_norm = _RMSNorm(d_model)
            self.head = nn.Sequential(
                nn.Linear(d_model, d_model),
                nn.SiLU(),
                nn.Linear(d_model, vocab_size),
            )
            self._backbone_name = "mamba"

        def forward(self, x, inference_params=None):
            h = self.embedding(x)                  # [B, L, D]
            for blk in self.blocks:
                h = blk(h, inference_params=inference_params)
            h = self.final_norm(h)
            return self.head(h)                    # [B, L, vocab_size]

        if IS_CUDA:
            @torch.inference_mode()
            def init_stream(self, max_len: int, batch_size: int = 1, device=None, dtype=None):
                return InferenceParams(max_batch_size=batch_size, max_seqlen=max_len)

            @torch.inference_mode()
            def step(self, byte_t: torch.LongTensor, inf) -> torch.Tensor:
                h = self.embedding(byte_t).unsqueeze(1)   # [B, 1, D]
                for blk in self.blocks:
                    h = blk(h, inference_params=inf)
                h = self.final_norm(h).squeeze(1)         # [B, D]
                logits_next = self.head(h)                 # [B, vocab_size]
                bump_offset(inf, 1)
                return logits_next
        else:
            @torch.inference_mode()
            def init_stream(self, max_len: int, batch_size: int = 1, device=None, dtype=None):
                return [blk.init_cache(batch_size, device) for blk in self.blocks]

            @torch.inference_mode()
            def step(self, byte_t: torch.LongTensor, caches) -> torch.Tensor:
                h = self.embedding(byte_t)                # [B, D]
                for i, blk in enumerate(self.blocks):
                    h, caches[i] = blk.step(h, caches[i])
                h = self.final_norm(h)
                return self.head(h)                       # [B, vocab_size]

    model = BoaBytePredictor(d_model=d_model, num_layers=num_layers, vocab_size=vocab_size)
    tag_mamba_layers_with_ids(model)
    return model

def _aligned_len(n_bytes: int, seq_len: int, batch_size: int) -> int:
    # number of usable bytes that fit whole (batch_size * seq_len) chunks
    block = seq_len * batch_size
    return (n_bytes // block) * block

def make_splits(data_bytes: bytes | np.ndarray, seq_len: int, batch_size: int,
                splits=(0.8, 0.1, 0.1)):
    assert abs(sum(splits) - 1.0) < 1e-6, "splits must sum to 1.0"
    buf = np.frombuffer(bytes(data_bytes), dtype=np.uint8)
    usable = _aligned_len(len(buf), seq_len, batch_size)
    buf = buf[:usable]

    n = len(buf)
    n_train = _aligned_len(int(n * splits[0]), seq_len, batch_size)
    n_val   = _aligned_len(int(n * splits[1]), seq_len, batch_size)
    n_test  = _aligned_len(n - n_train - n_val, seq_len, batch_size)

    i0, i1, i2 = 0, n_train, n_train + n_val
    train_bytes = buf[i0:i1].tobytes()
    val_bytes   = buf[i1:i2].tobytes()
    test_bytes  = buf[i2:i2+n_test].tobytes()

    return train_bytes, val_bytes, test_bytes

class ByteDataloader:
    """ Simple dataloader that yields batches of bytes. """
    def __init__(self, data_bytes, seq_len=1048576, batch_size=1, device="cuda"):
        self.data_bytes = np.frombuffer(data_bytes, dtype=np.uint8)
        self.seq_len = seq_len
        self.batch_size = batch_size
        self.pos = 0
        self.device = device
        # Pre-allocate a pinned CPU tensor and a GPU tensor to avoid
        # repeated allocation + async H2D copies each step.
        self._block = self.seq_len * self.batch_size
        if device == "cuda" and torch.cuda.is_available():
            self._cpu_buf = torch.empty(self.batch_size, self.seq_len,
                                        dtype=torch.long, pin_memory=True)
            self._gpu_buf = torch.empty(self.batch_size, self.seq_len,
                                        dtype=torch.long, device=device)
        else:
            self._cpu_buf = None
            self._gpu_buf = None
    def __len__(self):
        """ Returns the total number of batches in the dataset. """
        return len(self.data_bytes) // (self.seq_len * self.batch_size)
    def __iter__(self):
        return self
    def __next__(self):
        if self.pos + self._block > len(self.data_bytes):
            self.pos = 0  # reset for simplicity
            raise StopIteration

        chunk = self.data_bytes[self.pos : self.pos + self._block]
        self.pos += self._block

        if self._cpu_buf is not None:
            # Fast path: copy into pinned buffer, async transfer to GPU
            np.copyto(self._cpu_buf.numpy().ravel(), chunk)
            self._gpu_buf.copy_(self._cpu_buf, non_blocking=True)
            return self._gpu_buf
        else:
            batch = chunk.reshape(self.batch_size, self.seq_len)
            return torch.tensor(batch, dtype=torch.long).to(self.device)



Writing model.py


**`gpu_range_coder.py`** — CUDA range coder (built on first import).


In [7]:
%%writefile gpu_range_coder.py

from __future__ import annotations

import os
import sys
import sysconfig
import importlib
import importlib.util
import pathlib
import tempfile
from textwrap import dedent
import subprocess
import shutil

def _build_and_import_cuda_extension() -> object:
    try:
        import pybind11  # noqa: F401
    except Exception:
        raise RuntimeError("pybind11 is required to build CUDA extension")

    build_dir = pathlib.Path(tempfile.gettempdir()) / "gpu_range_build"
    build_dir.mkdir(parents=True, exist_ok=True)
    ext_name = "_gpu_range_cuda_ext"
    src_cu = build_dir / (ext_name + ".cu")

    # CUDA source implementing a constriction-compatible word-based Range Coder (u32 words, u64 state)
    cuda_code = dedent(r'''
    #include <pybind11/pybind11.h>
    #include <pybind11/numpy.h>
    #include <pybind11/stl.h>
    #include <cuda_runtime.h>
    #include <cstdint>
    #include <math.h>
    #include <vector>
    #include <stdexcept>
    #include <algorithm>
    #include <cstring>

    namespace py = pybind11;

    static constexpr int PRECISION = 24;

    struct EncState {
        unsigned long long lower;
        unsigned long long range;
        int inverted_num;
        unsigned int first_inv_lower_word;
        int write_idx_words;
    };

    struct DecState {
        unsigned long long lower;
        unsigned long long range;
        unsigned long long point;
        int read_idx_words;
    };

    __device__ void build_cdf_fast(const float* probs_row, int K, uint32_t* cdf) {
        const uint32_t TOTAL = 1u << PRECISION;
        if (K <= 0) { cdf[0] = 0; cdf[1] = TOTAL; return; }
        const uint32_t free_weight = TOTAL - (uint32_t)K;
        double norm = 0.0;
        for (int i = 0; i < K; ++i) norm += (double)probs_row[i];
        if (!(norm > 0.0) || !isfinite(norm)) {
            cdf[0] = 0;
            uint32_t acc = 0;
            for (int i=0;i<K;i++) { cdf[i] = acc; acc += (free_weight / (uint32_t)K) + 1u; }
            cdf[K] = TOTAL; return;
        }
        double scale = (double)free_weight / norm;
        double cumulative_float = 0.0;
        uint32_t accumulated_slack = 0;
        for (int i=0;i<K;i++) {
            uint32_t left = (uint32_t)(cumulative_float * scale) + accumulated_slack;
            cdf[i] = left;
            cumulative_float += (double)probs_row[i];
            accumulated_slack += 1u;
        }
        cdf[K] = TOTAL;
    }

    __device__ __forceinline__ void flush_inverted(uint32_t* out_row, EncState* st) {
        if (st->inverted_num > 0) {
            // We can't know wrap condition without original pre-update lower; use point at seal time.
            // This function is only called in finalize; handled there.
        }
    }

    __global__ void encode_kernel(const int32_t* d_symbols, const float* d_probs, int N, int K,
                                  uint32_t* d_out_words, EncState* states, int pitch_words,
                                  const uint8_t* d_mask) {
        int idx = blockIdx.x * blockDim.x + threadIdx.x;
        if (idx >= N) return;
        if (d_mask && d_mask[idx] == 0) return;
        EncState &st = states[idx];
        const int32_t s = d_symbols[idx];
        const float* probs_row = d_probs + (size_t)idx * (size_t)K;
        if (s < 0 || s >= K) return;
        uint32_t cdf[1025];
        build_cdf_fast(probs_row, K, cdf);
        uint32_t left = cdf[s];
        uint32_t prob = cdf[s+1] - cdf[s];
        unsigned long long scale = st.range >> PRECISION;
        st.range = scale * (unsigned long long)prob;
        unsigned long long old_lower = st.lower;
        st.lower = st.lower + scale * (unsigned long long)left;
        // Handle transition out of inverted
        if (st.inverted_num > 0) {
            unsigned long long sum = st.lower + st.range;
            if (sum > st.lower) {
                uint32_t first_word, subsequent;
                if (st.lower < old_lower) { first_word = st.first_inv_lower_word + 1u; subsequent = 0u; }
                else { first_word = st.first_inv_lower_word; subsequent = 0xFFFFFFFFu; }
                int widx = st.write_idx_words;
                uint32_t* out = d_out_words + (size_t)idx * (size_t)pitch_words;
                out[(size_t)widx++] = first_word;
                for (int i=1;i<st.inverted_num;i++) out[(size_t)widx++] = subsequent;
                st.write_idx_words = widx;
                st.inverted_num = 0;
            }
        }
        // Renormalize if needed (emit possibly multiple words)
        while (st.range < (1ull << (64-32))) {
            uint32_t lower_word = (uint32_t)(st.lower >> (64-32));
            st.lower <<= 32;
            st.range <<= 32;
            if (st.inverted_num > 0) {
                if (st.inverted_num < 0x7FFFFFFF) st.inverted_num += 1;
            } else {
                unsigned long long sum = st.lower + st.range;
                if (sum > st.lower) {
                    int widx = st.write_idx_words;
                    uint32_t* out = d_out_words + (size_t)idx * (size_t)pitch_words;
                    out[(size_t)widx++] = lower_word;
                    st.write_idx_words = widx;
                } else {
                    st.inverted_num = 1;
                    st.first_inv_lower_word = lower_word;
                }
            }
        }
    }

    __global__ void finalize_kernel(int N, uint32_t* d_out_words, EncState* states, int pitch_words, int* sizes_words) {
        int idx = blockIdx.x * blockDim.x + threadIdx.x;
        if (idx >= N) return;
        EncState &st = states[idx];
        uint32_t* out = d_out_words + (size_t)idx * (size_t)pitch_words;
        if (st.range == 0xFFFFFFFFFFFFFFFFull) { sizes_words[idx] = 0; return; }
        unsigned long long point = st.lower + ((1ull << (64-32)) - 1ull);
        if (st.inverted_num > 0) {
            uint32_t first_word, subsequent;
            if (point < st.lower) { first_word = st.first_inv_lower_word + 1u; subsequent = 0u; }
            else { first_word = st.first_inv_lower_word; subsequent = 0xFFFFFFFFu; }
            int widx = st.write_idx_words;
            out[(size_t)widx++] = first_word;
            for (int i=1;i<st.inverted_num;i++) out[(size_t)widx++] = subsequent;
            st.write_idx_words = widx;
            st.inverted_num = 0;
        }
        uint32_t point_word = (uint32_t)(point >> (64-32));
        int widx = st.write_idx_words;
        out[(size_t)widx++] = point_word;
        unsigned long long upper = st.lower + st.range;
        uint32_t upper_word = (uint32_t)(upper >> (64-32));
        if (upper_word == point_word) out[(size_t)widx++] = 0u;
        st.write_idx_words = widx;
        sizes_words[idx] = widx;
    }

    __global__ void init_dec_kernel(int N, const uint32_t* d_in_words, int* sizes_words, DecState* dst, int pitch_words) {
        int idx = blockIdx.x * blockDim.x + threadIdx.x;
        if (idx >= N) return;
        const uint32_t* in = d_in_words + (size_t)idx * (size_t)pitch_words;
        DecState &st = dst[idx];
        st.lower = 0ull; st.range = 0xFFFFFFFFFFFFFFFFull; st.point = 0ull; st.read_idx_words = 0;
        for (int i=0;i<2;i++) {
            uint32_t w = (st.read_idx_words < sizes_words[idx]) ? in[st.read_idx_words++] : 0u;
            st.point = (st.point << 32) | (unsigned long long)w;
        }
    }

    __global__ void decode_step_kernel(int N, int K, const float* d_probs, const uint32_t* d_in_words, int* sizes_words,
                                       DecState* st_arr, int pitch_words, int32_t* out_symbols, const uint8_t* d_mask) {
        int idx = blockIdx.x * blockDim.x + threadIdx.x;
        if (idx >= N) return;
        if (d_mask && d_mask[idx] == 0) return;
        DecState &st = st_arr[idx];
        const float* probs_row = d_probs + (size_t)idx * (size_t)K;
        const uint32_t* in = d_in_words + (size_t)idx * (size_t)pitch_words;
        if (K <= 0) { out_symbols[idx] = 0; return; }
        uint32_t cdf[1025];
        build_cdf_fast(probs_row, K, cdf);
        unsigned long long scale = st.range >> PRECISION;
        unsigned long long q = (st.point - st.lower) / scale;
        if (q >= (1ull<<PRECISION)) q = (1ull<<PRECISION)-1ull;
        uint32_t target = (uint32_t)q;
        int next_symbol = 1;
        while (next_symbol <= K && !(cdf[next_symbol] > target)) ++next_symbol;
        int s = next_symbol - 1; if (s < 0) s = 0; if (s >= K) s = K-1;
        out_symbols[idx] = (int32_t)s;
        uint32_t left = cdf[s]; uint32_t prob = cdf[s+1] - cdf[s];
        st.lower = st.lower + scale * (unsigned long long)left;
        st.range = scale * (unsigned long long)prob;
        while (st.range < (1ull << (64-32))) {
            st.lower <<= 32;
            st.range <<= 32;
            uint32_t w = (st.read_idx_words < sizes_words[idx]) ? in[st.read_idx_words++] : 0u;
            st.point = (st.point << 32) | (unsigned long long)w;
        }
    }

    class RangeCoderBatch {
    public:
        int N, K, pitch;
        EncState* d_enc_states;
        DecState* d_dec_states;
        uint32_t* d_words;
        int* d_sizes; // sizes in words
        RangeCoderBatch(int N_, int K_, int pitch_) : N(N_), K(K_), pitch(pitch_), d_enc_states(nullptr), d_dec_states(nullptr), d_words(nullptr), d_sizes(nullptr) {
            cudaMalloc(&d_enc_states, sizeof(EncState) * N);
            cudaMalloc(&d_dec_states, sizeof(DecState) * N);
            cudaMalloc(&d_words, (size_t)N * (size_t)pitch * sizeof(uint32_t));
            cudaMalloc(&d_sizes, sizeof(int) * N);
            std::vector<EncState> init(N);
            for (int i = 0; i < N; ++i) { init[i].lower=0ull; init[i].range=0xFFFFFFFFFFFFFFFFull; init[i].inverted_num=0; init[i].first_inv_lower_word=0u; init[i].write_idx_words=0; }
            cudaMemcpy(d_enc_states, init.data(), sizeof(EncState) * N, cudaMemcpyHostToDevice);
            std::vector<int> zeros(N, 0);
            cudaMemcpy(d_sizes, zeros.data(), sizeof(int) * N, cudaMemcpyHostToDevice);
            cudaMemset(d_words, 0, (size_t)N * (size_t)pitch * sizeof(uint32_t));
        }
        ~RangeCoderBatch() {
            cudaFree(d_enc_states);
            cudaFree(d_dec_states);
            cudaFree(d_words);
            cudaFree(d_sizes);
        }
        void load_compressed_from_host(py::list compressed_list) {
            if ((int)compressed_list.size() != N) throw std::runtime_error("compressed_list length must equal N");
            std::vector<int> sizes_host(N, 0);
            std::vector<uint32_t> buf((size_t)N * (size_t)pitch, 0u);
            for (int i=0;i<N;i++) {
                py::array arr = py::array::ensure(compressed_list[i]);
                if (!arr || arr.ndim()!=1 || arr.itemsize()!=4 || arr.dtype().kind()!='u') throw std::runtime_error("Each compressed item must be uint32[?]");
                size_t nwords = (size_t)arr.shape(0);
                if (nwords > (size_t)pitch) throw std::runtime_error("Compressed stream exceeds pitch; increase pitch");
                const uint32_t* src = static_cast<const uint32_t*>(arr.data());
                uint32_t* dst = buf.data() + (size_t)i * (size_t)pitch;
                std::memcpy(dst, src, nwords * sizeof(uint32_t));
                sizes_host[i] = (int)nwords;
            }
            cudaMemcpy(d_words, buf.data(), buf.size()*sizeof(uint32_t), cudaMemcpyHostToDevice);
            cudaMemcpy(d_sizes, sizes_host.data(), sizeof(int)*N, cudaMemcpyHostToDevice);
        }
        std::vector<int> get_sizes_host() {
            std::vector<int> sizes(N);
            cudaMemcpy(sizes.data(), d_sizes, sizeof(int)*N, cudaMemcpyDeviceToHost);
            return sizes;
        }
        void set_sizes_from_host(py::list sizes_list) {
            if ((int)sizes_list.size()!=N) throw std::runtime_error("sizes_list length must equal N");
            std::vector<int> sizes(N);
            for (int i=0;i<N;i++) sizes[i] = sizes_list[i].cast<int>();
            cudaMemcpy(d_sizes, sizes.data(), sizeof(int)*N, cudaMemcpyHostToDevice);
        }
        void encode_step_from_device(uint64_t symbols_ptr, uint64_t probs_ptr, uint64_t mask_ptr) {
            const int32_t* d_symbols = reinterpret_cast<const int32_t*>(symbols_ptr);
            const float* d_probs = reinterpret_cast<const float*>(probs_ptr);
            const uint8_t* d_mask = reinterpret_cast<const uint8_t*>(mask_ptr);
            int threads = 128; int blocks = (N + threads - 1) / threads;
            encode_kernel<<<blocks, threads>>>(d_symbols, d_probs, N, K, d_words, d_enc_states, pitch, d_mask);
            cudaDeviceSynchronize();
        }
        void finalize() {
            int threads = 128; int blocks = (N + threads - 1) / threads;
            finalize_kernel<<<blocks, threads>>>(N, d_words, d_enc_states, pitch, d_sizes);
            cudaDeviceSynchronize();
        }
        std::vector<py::array_t<uint32_t>> get_compressed_host() {
            std::vector<int> sizes(N);
            cudaMemcpy(sizes.data(), d_sizes, sizeof(int) * N, cudaMemcpyDeviceToHost);
            std::vector<uint32_t> buf((size_t)N * (size_t)pitch);
            cudaMemcpy(buf.data(), d_words, buf.size()*sizeof(uint32_t), cudaMemcpyDeviceToHost);
            std::vector<py::array_t<uint32_t>> out; out.reserve(N);
            for (int i=0;i<N;i++) {
                size_t nwords = (size_t)std::max(0, sizes[i]);
                py::array_t<uint32_t> arr(nwords);
                auto r = arr.mutable_unchecked<1>();
                for (size_t w=0; w<nwords; ++w) r(w) = buf[(size_t)i*(size_t)pitch + w];
                out.push_back(arr);
            }
            return out;
        }
        void init_decoder_from_current_bytes() {
            int threads = 128; int blocks = (N + threads - 1) / threads;
            init_dec_kernel<<<blocks, threads>>>(N, d_words, d_sizes, d_dec_states, pitch);
            cudaDeviceSynchronize();
        }
        void decode_step_to_device(uint64_t probs_ptr, uint64_t out_symbols_ptr, uint64_t mask_ptr) {
            const float* d_probs = reinterpret_cast<const float*>(probs_ptr);
            int32_t* d_out = reinterpret_cast<int32_t*>(out_symbols_ptr);
            const uint8_t* d_mask = reinterpret_cast<const uint8_t*>(mask_ptr);
            int threads = 128; int blocks = (N + threads - 1) / threads;
            decode_step_kernel<<<blocks, threads>>>(N, K, d_probs, d_words, d_sizes, d_dec_states, pitch, d_out, d_mask);
            cudaDeviceSynchronize();
        }
    };

    PYBIND11_MODULE(_gpu_range_cuda_ext, m) {
        m.doc() = "GPU-backed range coder (constriction-compatible, u32 words, u64 state)";
        py::class_<RangeCoderBatch>(m, "RangeCoderBatch")
            .def(py::init<int,int,int>())
            .def("load_compressed_from_host", &RangeCoderBatch::load_compressed_from_host,
                 py::arg("compressed_list"))
            .def("get_sizes_host", &RangeCoderBatch::get_sizes_host)
            .def("set_sizes_from_host", &RangeCoderBatch::set_sizes_from_host,
                 py::arg("sizes_list"))
            .def("encode_step_from_device", &RangeCoderBatch::encode_step_from_device,
                 py::arg("symbols_ptr"), py::arg("probs_ptr"), py::arg("mask_ptr") = 0)
            .def("finalize", &RangeCoderBatch::finalize)
            .def("get_compressed_host", &RangeCoderBatch::get_compressed_host)
            .def("init_decoder_from_current_bytes", &RangeCoderBatch::init_decoder_from_current_bytes)
            .def("decode_step_to_device", &RangeCoderBatch::decode_step_to_device,
                 py::arg("probs_ptr"), py::arg("out_symbols_ptr"), py::arg("mask_ptr") = 0);
    }
    ''')

    src_cu.write_text(cuda_code)

    # Compile with nvcc
    nvcc = shutil.which('nvcc')
    if nvcc is None:
        raise RuntimeError('nvcc not found in PATH')
    from importlib.machinery import EXTENSION_SUFFIXES
    suf = EXTENSION_SUFFIXES[0]
    so_path = build_dir / (ext_name + suf)
    # Build include dirs robustly (handle spaces in paths)
    import pybind11
    inc_dirs = []
    try:
        cfg_paths = sysconfig.get_paths()
        for key in ('include', 'platinclude'):  # e.g., Python.h location
            p = cfg_paths.get(key)
            if p:
                inc_dirs.append(p)
    except Exception:
        pass
    # pybind11 headers
    for k in (pybind11.get_include(False), pybind11.get_include(True)):
        if k and k not in inc_dirs:
            inc_dirs.append(k)

    cmd = [nvcc, str(src_cu), '-shared', '-Xcompiler', '-fPIC', '-o', str(so_path),
           '-std=c++14', '-O3']
    # CUDA runtime library
    cuda_home = os.environ.get('CUDA_HOME') or os.environ.get('CUDA_PATH') or '/usr/local/cuda'
    lib64 = pathlib.Path(cuda_home) / 'lib64'
    if lib64.exists():
        cmd += ['-L', str(lib64)]
        # add rpath for runtime loading
        cmd += ['-Xlinker', f'-rpath,{str(lib64)}']
    cmd += ['-lcudart']
    # Add include directories as separate args to preserve spaces
    for inc in inc_dirs:
        cmd += ['-I', inc]
    # Run nvcc
    res = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    if res.returncode != 0:
        raise RuntimeError(f'nvcc build failed: {res.stderr.decode()}')

    spec = importlib.util.spec_from_file_location(ext_name, str(so_path))
    if spec is None or spec.loader is None:
        raise RuntimeError('Failed to load CUDA built extension')
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)  # type: ignore
    return mod


# Build extension on import: try CUDA if requested, otherwise CPU fallback
_ext = None
_cpu_ext = None  # optional CPU extension placeholder for fallbacks
_ext = _build_and_import_cuda_extension()

# Expose a minimal namespace compatible with the used subset: constriction.stream.queue
class _ModelStub:
    def __init__(self, kind: str, **kwargs):
        self.kind = kind
        self.kwargs = kwargs


class stream:
    class model:
        class Categorical(_ModelStub):
            def __init__(self, perfect: bool = False):
                super().__init__("categorical", perfect=perfect)

    class queue:
        class RangeEncoder:
            def __init__(self):
                # If extension provides a RangeEncoder class use it, otherwise prepare Python buffer for GPU function
                if hasattr(_ext, 'RangeEncoder'):
                    self._enc = _ext.RangeEncoder()
                    self._pybuf = None
                else:
                    self._enc = None
                    self._pybuf = {'symbols': [], 'probs': []}

            def clear(self):
                if self._enc is not None:
                    self._enc.clear()
                else:
                    self._pybuf = {'symbols': [], 'probs': []}

            def get_compressed(self):
                if self._enc is not None:
                    return self._enc.get_compressed()
                else:
                    import numpy as np
                    if len(self._pybuf['symbols']) == 0:
                        return np.zeros(0, dtype=np.uint32)
                    symbols = np.asarray(self._pybuf['symbols'], dtype=np.int32)
                    probs = np.asarray(self._pybuf['probs'], dtype=np.float32)
                    # For correctness, delegate to CPU extension's RangeEncoder to produce
                    # the exact same compressed format the CPU decoder expects.
                    if _cpu_ext is not None and hasattr(_cpu_ext, 'RangeEncoder'):
                        cpu_enc = _cpu_ext.RangeEncoder()
                        cpu_enc.encode_categorical(symbols, probs)
                        return cpu_enc.get_compressed()
                    # Fallback: try CUDA encode function (may be incompatible)
                    return _ext.encode_rows_gpu(symbols, probs)

            def encode(self, symbols, model, probs):
                """
                Supports Option 3: encode(symbols, model_family=Categorical, probs).
                - symbols: rank-1 int32 numpy array (len=N)
                - model: constriction.stream.model.Categorical (ignored other than type)
                - probs: rank-2 float32 numpy array with shape (N, K)
                """
                import numpy as np
                # Normalize inputs
                if not hasattr(symbols, "dtype"):
                    symbols = np.array([int(symbols)], dtype=np.int32)
                symbols = np.asarray(symbols, dtype=np.int32)
                probs = np.asarray(probs, dtype=np.float32)
                if symbols.ndim != 1:
                    raise ValueError("symbols must be rank-1")
                if probs.ndim != 2 or probs.shape[0] != symbols.shape[0]:
                    raise ValueError("probs must be rank-2 with probs.shape[0] == len(symbols)")
                if not isinstance(model, _ModelStub) or model.kind != "categorical":
                    raise TypeError("Only Categorical model is supported in this GPU stub")
                if self._enc is not None:
                    # delegate to compiled RangeEncoder
                    self._enc.encode_categorical(symbols, probs)
                else:
                    # buffer for batched GPU encode
                    for s in symbols.tolist():
                        self._pybuf['symbols'].append(int(s))
                    # ensure probs is list of rows
                    for row in probs.astype(np.float32):
                        self._pybuf['probs'].append(row.tolist())

        class RangeDecoder:
            def __init__(self, compressed):
                # Prefer compiled decoder if provided; otherwise fall back to CPU extension
                if hasattr(_ext, 'RangeDecoder'):
                    self._dec = _ext.RangeDecoder(compressed)
                elif _cpu_ext is not None and hasattr(_cpu_ext, 'RangeDecoder'):
                    self._dec = _cpu_ext.RangeDecoder(compressed)
                else:
                    raise RuntimeError('No decoder available in extensions')

            def decode(self, model, probs_or_amt, *rest):
                """
                Supports Option 3: decode(model_family=Categorical, probs) -> symbols array.
                """
                import numpy as np
                if not isinstance(model, _ModelStub) or model.kind != "categorical":
                    raise TypeError("Only Categorical model is supported in this GPU stub")
                probs = np.asarray(probs_or_amt, dtype=np.float32)
                if probs.ndim != 2:
                    raise ValueError("probs must be rank-2")
                return self._dec.decode_categorical(probs)


# For convenience, re-export top-level like constriction
__all__ = ["stream"]

# Optional: GPU batch coder convenience wrapper (requires torch)
class gpu:
    class queue:
        class RangeCoderBatch:
            def __init__(self, N: int, K: int, maxL: int, pitch_bytes: int | None = None):
                if pitch_bytes is None:
                    pitch_bytes = max(256, maxL * 8)
                self.N, self.K, self.maxL = N, K, maxL
                global _ext
                if _ext is None or not hasattr(_ext, 'RangeCoderBatch'):
                    # Try to build CUDA extension lazily if not present
                    try:
                        _ext = _build_and_import_cuda_extension()
                    except Exception as e:
                        raise RuntimeError('CUDA extension with RangeCoderBatch not available') from e
                # CUDA backend expects pitch in 32-bit words
                pitch_words = (int(pitch_bytes) + 3) // 4
                self._batch = _ext.RangeCoderBatch(N, K, pitch_words)

            def load_compressed_list(self, compressed_list):
                # Accept list of np.uint32 arrays (one per stream)
                self._batch.load_compressed_from_host(compressed_list)

            def encode_step(self, symbols_gpu, probs_gpu, mask=None):
                import torch
                assert symbols_gpu.is_cuda and probs_gpu.is_cuda
                assert symbols_gpu.numel() == self.N and probs_gpu.shape == (self.N, self.K)
                if symbols_gpu.dtype != torch.int32:
                    symbols_gpu = symbols_gpu.to(torch.int32)
                if probs_gpu.dtype != torch.float32:
                    probs_gpu = probs_gpu.to(torch.float32)
                mask_ptr = 0
                if mask is not None:
                    assert mask.is_cuda and mask.shape == (self.N,)
                    if mask.dtype != torch.uint8:
                        mask = mask.to(torch.uint8)
                    mask_ptr = int(mask.data_ptr())
                self._batch.encode_step_from_device(int(symbols_gpu.data_ptr()), int(probs_gpu.data_ptr()), mask_ptr)

            def finalize(self):
                self._batch.finalize()

            def get_compressed_list(self):
                return self._batch.get_compressed_host()

            def get_sizes_list(self):
                return self._batch.get_sizes_host()

            def init_decoder(self):
                self._batch.init_decoder_from_current_bytes()

            def decode_step(self, probs_gpu, out_symbols_gpu, mask=None):
                import torch
                assert probs_gpu.is_cuda and out_symbols_gpu.is_cuda
                assert probs_gpu.shape == (self.N, self.K) and out_symbols_gpu.numel() == self.N
                if probs_gpu.dtype != torch.float32:
                    probs_gpu = probs_gpu.to(torch.float32)
                mask_ptr = 0
                if mask is not None:
                    assert mask.is_cuda and mask.shape == (self.N,)
                    if mask.dtype != torch.uint8:
                        mask = mask.to(torch.uint8)
                    mask_ptr = int(mask.data_ptr())
                self._batch.decode_step_to_device(int(probs_gpu.data_ptr()), int(out_symbols_gpu.data_ptr()), mask_ptr)


Writing gpu_range_coder.py


**`codec.py`** — model-driven compress/decompress (GPU coder ↔ constriction).


In [8]:
%%writefile codec.py
import numpy as np
import torch
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm

IS_CUDA = torch.cuda.is_available()

# Colab robustness: building the GPU range coder needs nvcc. If it is not
# available (e.g. a CPU-only runtime) fall back to the constriction CPU codec
# instead of failing at import time.
gr = None
if IS_CUDA:
    try:
        import gpu_range_coder as gr  # builds a CUDA extension via nvcc on first import
    except Exception as _gpu_err:  # noqa: BLE001
        print(f'[codec] GPU range coder unavailable -> using CPU (constriction) codec: {_gpu_err}')
        IS_CUDA = False

if IS_CUDA:
    @torch.inference_mode()
    def compress_GPU(
        model, x_list: list[torch.Tensor], device="cuda", progress=True, num_workers: int = 8
    ):
        """
        x_list: list of N tensors, each shaped [1, L_i] with uint8 in [0..255]
        Returns:
        compressed_list: list[np.ndarray(uint32)]
        first_bytes: list[int]
        lengths: list[int]
        """
        # Setup
        model.eval().to(device)
        N = len(x_list)
        assert N >= 1, "Need at least one chunk."
        xs = [x.to(device, dtype=torch.long, non_blocking=True) for x in x_list]
        for i, x in enumerate(xs):
            assert x.ndim == 2 and x.shape[0] == 1, f"Chunk {i} must be [1, L_i]"
            assert x.shape[1] >= 1, f"Chunk {i} must have length >= 1"

        Ls = [int(x.shape[1]) for x in xs]
        maxL = max(Ls)

        # Pack into one [N, maxL] for batched reads (on GPU)
        X = torch.zeros((N, maxL), dtype=torch.long, device=device)
        for i, x in enumerate(xs):
            X[i, :Ls[i]] = x[0]

        first_bytes = X[:, 0].tolist()
        lens_t = torch.tensor(Ls, device=device, dtype=torch.long)

        # GPU batch range encoder (no D2H for probs/symbols)
        # K = vocab_size for bytes
        vocab_size = model.embedding.num_embeddings
        batch = gr.gpu.queue.RangeCoderBatch(N, vocab_size, maxL)

        # Streaming state
        inf = model.init_stream(max_len=maxL, batch_size=N, device=device, dtype=torch.float32)
        prev = X[:, 0].clone()  # [N] device

        total_steps = sum(L - 1 for L in Ls)
        pbar = tqdm(total=total_steps, disable=not progress, desc=f"Compress (GPU streams x{N})",
                    unit="KB", unit_scale=1/1024, mininterval=0.2)

        # Encode timesteps t = 1..maxL-1
        for t in range(1, maxL):
            # Active lanes this step
            lens_mask = (lens_t > t)  # [N] bool on device
            if not torch.any(lens_mask):
                break

            # Compute probabilities on GPU for current prev
            logits = model.step(prev, inf)
            if logits.ndim == 3:
                logits = logits.squeeze(1)
            probs_gpu = torch.softmax(logits, dim=-1).to(torch.float32)

            # Symbols to encode this step (on GPU)
            syms = X[:, t].to(dtype=torch.int32)

            # Encode on GPU (masked; inactive lanes are skipped)
            # Note: encode_step supports optional mask: torch.bool [N]
            batch.encode_step(syms, probs_gpu, mask=lens_mask)

            # Update prev only for active lanes
            prev = torch.where(lens_mask, X[:, t], prev)

            # Progress: number of lanes still active at step t
            pbar.update(int(lens_mask.sum().item()))

        pbar.close()

        # Finalize on GPU and bring compressed outputs back as np.uint32 lists
        batch.finalize()
        compressed_list = batch.get_compressed_list()
        return compressed_list, first_bytes, Ls


    def decompress_GPU(
        model, compressed_list, full_lens: list[int], first_bytes: list[int],
        device="cuda", progress=True, num_workers: int = 8
    ):
        """
        Returns: list[np.ndarray] each (1, L_i) uint8
        """
        with torch.inference_mode():
            model.eval().to(device)
            N = len(compressed_list)
            assert N >= 1 and len(full_lens) == N and len(first_bytes) == N
            assert all(L >= 1 for L in full_lens)

            maxL = max(full_lens)
            lens_t = torch.tensor(full_lens, device=device, dtype=torch.long)

            # Initialize GPU batch decoder from compressed streams (no D2H for probs)
            vocab_size = model.embedding.num_embeddings
            dec = gr.gpu.queue.RangeCoderBatch(N, vocab_size, maxL)

            # Output buffer fully on GPU; we copy to host only at the end
            outs_gpu = torch.empty((N, maxL), dtype=torch.uint8, device=device)
            outs_gpu[:, 0] = torch.as_tensor(first_bytes, device=device, dtype=torch.uint8)

            # Streaming state
            inf = model.init_stream(max_len=maxL, batch_size=N, device=device, dtype=torch.float32)
            prev = torch.as_tensor(first_bytes, dtype=torch.long, device=device)

            total_steps = sum(L - 1 for L in full_lens)
            pbar = tqdm(total=total_steps, disable=not progress, desc=f"Decompress (GPU streams x{N})",
                        unit="KB", unit_scale=1/1024, mininterval=0.2)

            dec.load_compressed_list(compressed_list)
            dec.init_decoder()
            # Decode timesteps t = 1..maxL-1
            out_syms = torch.empty((N,), dtype=torch.int32, device=device)
            for t in range(1, maxL):
                lens_mask = (lens_t > t)
                if not torch.any(lens_mask):
                    break

                # Compute probabilities on GPU for current prev
                logits = model.step(prev, inf)
                if logits.ndim == 3:
                    logits = logits.squeeze(1)
                probs_gpu = torch.softmax(logits, dim=-1).to(torch.float32)

                # Decode on GPU into out_syms (masked lanes only)
                dec.decode_step(probs_gpu, out_syms, mask=lens_mask)

                # Write decoded symbols for active lanes and update prev
                outs_gpu[lens_mask, t] = out_syms[lens_mask].to(torch.uint8)
                prev = torch.where(lens_mask, out_syms.to(torch.long), prev)

                pbar.update(int(lens_mask.sum().item()))

            pbar.close()

            # Materialize outputs on host once
            outs = []
            for i in range(N):
                outs.append(outs_gpu[i, :full_lens[i]].detach().to("cpu").numpy().reshape(1, -1))
            return outs

@torch.inference_mode()
def compress_CPU(
    model, x_list: list[torch.Tensor], device="cpu", progress=True, num_workers: int = 8
):
    """
    x_list: list of N tensors, each shaped [1, L_i] with uint8 in [0..255]
    Returns:
      compressed_list: list[np.ndarray(uint32)]
      first_bytes: list[int]
      lengths: list[int]
    """
    # Setup (CPU only)
    device = "cpu"
    model.eval().to(device)
    N = len(x_list)
    assert N >= 1, "Need at least one chunk."
    xs = [x.to(device, dtype=torch.long) for x in x_list]
    for i, x in enumerate(xs):
        assert x.ndim == 2 and x.shape[0] == 1, f"Chunk {i} must be [1, L_i]"
        assert x.shape[1] >= 1, f"Chunk {i} must have length >= 1"

    Ls = [int(x.shape[1]) for x in xs]
    maxL = max(Ls)

    # Pack into one [N, maxL] tensor on CPU
    X = torch.zeros((N, maxL), dtype=torch.long, device=device)
    for i, x in enumerate(xs):
        X[i, :Ls[i]] = x[0]
    first_bytes = X[:, 0].tolist()

    import constriction
    fam = constriction.stream.model.Categorical(perfect=False)
    encs = [constriction.stream.queue.RangeEncoder() for _ in range(N)]

    # Streaming state (CPU cache structure)
    caches = model.init_stream(max_len=maxL, batch_size=N, device=device, dtype=torch.float32)
    prev = X[:, 0].clone()  # [N] CPU

    total_steps = sum(L - 1 for L in Ls)
    pbar = tqdm(total=total_steps, disable=not progress, desc=f"Compress (CPU streams x{N})",
                unit="KB", unit_scale=1/1024, mininterval=0.2)

    X_cpu = X.detach().cpu().numpy().astype(np.int32, copy=False)

    def encode_range(r0, r1, t, probs_np):
        for i in range(r0, r1):
            if t < Ls[i]:
                sym = int(X_cpu[i, t])
                encs[i].encode(np.array([sym], dtype=np.int32), fam, probs_np[i:i+1, :])

    for t in range(1, maxL):
        # Determine active lanes
        active = np.asarray(Ls) > t
        if not active.any():
            break

        # Compute probabilities on CPU for current prev
        logits = model.step(prev, caches)
        if logits.ndim == 3:
            logits = logits.squeeze(1)
        probs = torch.softmax(logits, dim=-1).to(torch.float32)
        probs_np = probs.detach().cpu().numpy()

        # Parallel lane-wise encoding on CPU
        if num_workers and num_workers > 1:
            chunk = (N + num_workers - 1) // num_workers
            futs = []
            with ThreadPoolExecutor(max_workers=num_workers) as pool:
                s = 0
                while s < N:
                    e = min(s + chunk, N)
                    futs.append(pool.submit(encode_range, s, e, t, probs_np))
                    s = e
                for f in futs:
                    f.result()
        else:
            encode_range(0, N, t, probs_np)

        # Update prev only for active lanes
        lens_mask = torch.from_numpy(active).to(torch.bool)
        prev = torch.where(lens_mask, X[:, t], prev)

        pbar.update(int(active.sum()))

    pbar.close()
    compressed_list = [encs[i].get_compressed() for i in range(N)]
    return compressed_list, first_bytes, Ls


def decompress_CPU(
    model, compressed_list, full_lens: list[int], first_bytes: list[int],
    device="cpu", progress=True, num_workers: int = 8
):
    """
    Returns: list[np.ndarray] each (1, L_i) uint8
    """
    # CPU-only implementation
    device = "cpu"
    with torch.inference_mode():
        model.eval().to(device)
        N = len(compressed_list)
        assert N >= 1 and len(full_lens) == N and len(first_bytes) == N
        assert all(L >= 1 for L in full_lens)

        maxL = max(full_lens)

        def as_u32(comp):
            if isinstance(comp, np.ndarray) and comp.dtype == np.uint32:
                return comp
            elif isinstance(comp, np.ndarray) and comp.dtype == np.uint8:
                return comp.view(np.uint32)
            else:
                return np.frombuffer(np.asarray(comp).tobytes(), dtype=np.uint32)

        import constriction
        fam = constriction.stream.model.Categorical(perfect=False)
        decs = [constriction.stream.queue.RangeDecoder(as_u32(compressed_list[i])) for i in range(N)]
        outs = [np.empty(full_lens[i], dtype=np.uint8) for i in range(N)]
        for i in range(N):
            outs[i][0] = int(first_bytes[i])

        # Streaming state (CPU caches)
        caches = model.init_stream(max_len=maxL, batch_size=N, device=device, dtype=torch.float32)
        prev = torch.tensor(first_bytes, dtype=torch.long, device=device)

        total_steps = sum(L - 1 for L in full_lens)
        pbar = tqdm(total=total_steps, disable=not progress, desc=f"Decompress (CPU streams x{N})",
                    unit="KB", unit_scale=1/1024, mininterval=0.2)
        lens_arr = np.asarray(full_lens, dtype=np.int64)

        def decode_range(r0, r1, t, probs_np, prev_np):
            for i in range(r0, r1):
                if t < full_lens[i]:
                    sym = int(decs[i].decode(fam, probs_np[i:i+1, :])[0])
                    outs[i][t] = sym
                    prev_np[i] = sym

        prev_np = np.array(first_bytes, dtype=np.int32)

        for t in range(1, maxL):
            active = lens_arr > t
            if not active.any():
                break

            logits = model.step(prev, caches)
            if logits.ndim == 3:
                logits = logits.squeeze(1)
            probs = torch.softmax(logits, dim=-1).to(torch.float32)
            probs_np = probs.detach().cpu().numpy()

            # Parallel lane-wise decode on CPU
            if num_workers and num_workers > 1:
                chunk = (N + num_workers - 1) // num_workers
                futs, s = [], 0
                with ThreadPoolExecutor(max_workers=num_workers) as pool:
                    while s < N:
                        e = min(s + chunk, N)
                        futs.append(pool.submit(decode_range, s, e, t, probs_np, prev_np))
                        s = e
                    for f in futs:
                        f.result()
            else:
                decode_range(0, N, t, probs_np, prev_np)

            # Update prev tensor from numpy buffer for next step
            prev = torch.from_numpy(prev_np).to(torch.long)

            pbar.update(int(active.sum()))

        pbar.close()
        return [o.reshape(1, -1) for o in outs]


Writing codec.py


**`boa.py`** — `.boa` container, chunking, compress/decompress entry points.


In [9]:
%%writefile boa.py
from pathlib import Path
import struct, zlib, math, hashlib
import os
import numpy as np
import torch

# Online head adaptation (test-time training of prediction head)
try:
    from online_adapt import HeadAdapter
    _HAS_ADAPT = True
except ImportError:
    _HAS_ADAPT = False


def BOA(device, filepath: str, model, adapt_config=None, measure_energy=False):
    IS_CUDA = device == "cuda" and torch.cuda.is_available()

    if IS_CUDA:
        from codec import compress_GPU as compress, decompress_GPU as decompress
        device = "cuda"
    else:
        from codec import compress_CPU as compress, decompress_CPU as decompress
        device = "cpu"

    # Energy measurement setup
    _energy_tracker = None
    if measure_energy:
        try:
            from energy_tracker import EnergyTracker as _ET
            _energy_tracker = _ET
        except Exception as e:
            print(f"[WARN] --measure-energy requested but energy_tracker unavailable: {e}")
            print("       Build CPPJoules first (see portability_solved_cpp/CPPJoules/)")
            _energy_tracker = None

    def _uvarint_encode(x: int) -> bytes:
        out = bytearray()
        while True:
            b = x & 0x7F; x >>= 7
            out.append(b | (0x80 if x else 0))
            if not x: break
        return bytes(out)

    def _uvarint_decode(buf: memoryview, pos: int):
        x = 0; s = 0
        while True:
            b = buf[pos]; pos += 1
            x |= (b & 0x7F) << s
            if not (b & 0x80): return x, pos
            s += 7

    def _as_bytes(obj) -> bytes:
        if isinstance(obj, (bytes, bytearray)): return bytes(obj)
        if torch.is_tensor(obj):
            t = obj.detach().contiguous().to("cpu")
            if t.dtype != torch.uint8: t = t.view(torch.uint8)
            return t.numpy().tobytes()
        arr = np.asarray(obj)
        if arr.dtype != np.uint8: arr = arr.view(np.uint8)
        return arr.tobytes()

    def _pad4(b: bytes) -> bytes:
        r = len(b) & 3
        return b if r == 0 else (b + b"\x00" * (4 - r))

    class BoaFile:
        MAGIC = b'BOA2'
        IDX   = b'IDX2'
        VERSION = 1

        def __init__(self, filepath: str, model):
            self.filepath = Path(filepath)
            self.model = model
            self.compressed_data = []
            self.first_bytes = []
            self.lengths = []
            self.metadata = {}

        def _split_to_chunks(self, data_bytes: bytes, seq_size: int = 0, chunks_count: int = 0):
            n = len(data_bytes)
            if seq_size and chunks_count:
                chunk_len = int(seq_size); n_chunks = math.ceil(n / chunk_len)
            elif seq_size:
                chunk_len = int(seq_size); n_chunks = math.ceil(n / chunk_len)
            elif chunks_count:
                n_chunks = max(int(chunks_count), 1); chunk_len = math.ceil(n / n_chunks)
            else:
                raise ValueError("Provide either 'seq_size' or 'chunks_count'.")
            chunks = []
            for i in range(n_chunks):
                s = i * chunk_len; e = min(s + chunk_len, n)
                if s >= e: break
                arr = np.frombuffer(memoryview(data_bytes)[s:e], dtype=np.uint8).astype(np.int64)
                chunks.append(arr)
            last_len = len(chunks[-1]) if chunks else 0
            self.metadata = {
                'chunk_len': int(chunk_len),
                'n_chunks': int(len(chunks)),
                'uncompressed_len': int(n),
                'last_chunk_len': int(last_len if last_len else chunk_len),
            }
            return chunks, int(chunk_len)

        def _model_fingerprint(self) -> bytes:
            name = getattr(self.model, "__class__", type(self.model)).__name__
            return hashlib.blake2s(name.encode(), digest_size=16).digest()

        def _write_file(self, compressed_list, first_bytes, uncompressed_len, chunk_len, last_chunk_len):
            n = len(compressed_list); fp = self._model_fingerprint()
            with open(self.filepath, 'wb') as f:
                f.write(self.MAGIC)
                f.write(struct.pack('<I', self.VERSION))
                f.write(struct.pack('<I', 0))  # flags
                f.write(struct.pack('<Q', uncompressed_len))
                f.write(struct.pack('<I', chunk_len))
                f.write(struct.pack('<I', n))
                f.write(struct.pack('<I', last_chunk_len))
                f.write(struct.pack('<B', len(fp))); f.write(fp)

                offsets = []; off = 0
                for c in compressed_list:
                    offsets.append(off); f.write(c); off += len(c)

                idx = bytearray()
                idx += self.IDX
                idx += bytes(first_bytes)  # n bytes
                # IDX2: only lengths (offsets reconstructed as cumulative sum)
                for L in lengths:
                    idx += _uvarint_encode(L)
                crc = zlib.crc32(idx) & 0xFFFFFFFF
                f.write(idx); f.write(struct.pack('<I', crc))

        def _read_file(self):
            data = Path(self.filepath).read_bytes()
            mm = memoryview(data); p = 0
            if bytes(mm[p:p+4]) != self.MAGIC: raise ValueError("Bad file magic")
            p += 4
            version, = struct.unpack_from('<I', mm, p); p += 4
            if version != self.VERSION: raise ValueError(f"Unsupported version {version}")
            _flags, = struct.unpack_from('<I', mm, p); p += 4
            ulen, = struct.unpack_from('<Q', mm, p); p += 8
            chunk_len, = struct.unpack_from('<I', mm, p); p += 4
            n, = struct.unpack_from('<I', mm, p); p += 4
            last_chunk_len, = struct.unpack_from('<I', mm, p); p += 4
            hlen = mm[p]; p += 1
            _fp = bytes(mm[p:p+hlen]); p += hlen

            crc = struct.unpack_from('<I', mm, len(mm)-4)[0]
            # Support both IDX1 and IDX2
            idx_pos_2 = data.rfind(b'IDX2')
            idx_pos_1 = data.rfind(b'IDX1')
            idx_pos = max(idx_pos_2, idx_pos_1)
            if idx_pos < 0: raise ValueError("Index not found")
            idx_ver = 2 if idx_pos == idx_pos_2 else 1
            if (zlib.crc32(data[idx_pos:len(data)-4]) & 0xFFFFFFFF) != crc:
                raise ValueError("Bad index CRC")

            q = idx_pos + 4  # skip IDXn tag
            first_bytes = list(mm[q:q+n]); q += n

            if idx_ver == 1:
                offsets = [0]*n; pos = q; prev = 0
                for i in range(n):
                    d, pos = _uvarint_decode(mm, pos); prev += d; offsets[i] = prev
                comp_lens = [0]*n
                for i in range(n):
                    L, pos = _uvarint_decode(mm, pos); comp_lens[i] = L
            else:
                # IDX2: only lengths, reconstruct offsets
                comp_lens = [0]*n; pos = q
                for i in range(n):
                    L, pos = _uvarint_decode(mm, pos); comp_lens[i] = L
                offsets = [0]*n; off = 0
                for i in range(n):
                    offsets[i] = off; off += comp_lens[i]

            payload = mm[p:idx_pos]
            compressed_list = [bytes(payload[offsets[i]: offsets[i]+comp_lens[i]]) for i in range(n)]
            full_lens = [int(chunk_len)]*(n-1) + [int(last_chunk_len)]

            self.compressed_data = compressed_list          # bytes
            self.first_bytes = first_bytes                  # list[int]
            self.lengths = full_lens                        # uncompressed lens
            self.metadata = {
                'chunk_len': int(chunk_len),
                'n_chunks': int(n),
                'last_chunk_len': int(last_chunk_len),
                'uncompressed_len': int(ulen),
            }

        def compress(self, data_path: str, seq_size: int = 0, chunks_count: int = 0, progress: bool = True):
            # Energy tracking
            _tracker = None
            if _energy_tracker is not None:
                _tracker = _energy_tracker()
                _tracker.start()
                if progress:
                    print("[energy] Started energy measurement for compression")

            # Determine chunking from file size without loading entire file into RAM
            p = Path(data_path)
            total_size = p.stat().st_size
            if total_size <= 0:
                raise ValueError("Input file is empty")

            # Compute chunk_len and number of chunks similar to _split_to_chunks
            if seq_size and chunks_count:
                chunk_len = int(seq_size); n_chunks = math.ceil(total_size / chunk_len)
            elif seq_size:
                chunk_len = int(seq_size); n_chunks = math.ceil(total_size / chunk_len)
            elif chunks_count:
                n_chunks = max(int(chunks_count), 1); chunk_len = math.ceil(total_size / n_chunks)
                n_chunks = math.ceil(total_size / chunk_len)  # actual #chunks; avoids overshoot -> negative last_chunk_len
            else:
                raise ValueError("Provide either 'seq_size' or 'chunks_count'.")

            last_chunk_len = int(total_size - (n_chunks - 1) * chunk_len) if n_chunks > 1 else int(total_size)
            self.metadata = {
                'chunk_len': int(chunk_len),
                'n_chunks': int(n_chunks),
                'uncompressed_len': int(total_size),
                'last_chunk_len': int(last_chunk_len if last_chunk_len else chunk_len),
            }

            # Prepare output file and write header
            fp = self._model_fingerprint()
            with open(self.filepath, 'wb') as f_out:
                f_out.write(self.MAGIC)
                f_out.write(struct.pack('<I', self.VERSION))
                f_out.write(struct.pack('<I', 0))  # flags
                f_out.write(struct.pack('<Q', total_size))
                f_out.write(struct.pack('<I', chunk_len))
                f_out.write(struct.pack('<I', n_chunks))
                f_out.write(struct.pack('<I', last_chunk_len))
                f_out.write(struct.pack('<B', len(fp))); f_out.write(fp)

                payload_start = f_out.tell()
                offsets: list[int] = []
                lengths: list[int] = []
                first_bytes: list[int] = []
                off = 0

                # Memory-map the input file to avoid loading it into RAM
                mm = np.memmap(p, dtype=np.uint8, mode='r')

                # Number of chunks to process per batch (streams). Default 5000; can be overridden for demos via env.
                try:
                    gpu_streams = int(os.getenv("BOA_GPU_STREAMS", "5000"))
                except Exception:
                    gpu_streams = 5000
                gpu_streams = max(1, min(int(gpu_streams), int(n_chunks)))
                if progress:
                    try:
                        if device == "cuda" and torch.cuda.is_available():
                            free_mem, total_mem = torch.cuda.mem_get_info()
                            print(f"[compress] gpu_streams={gpu_streams} (chunk_len={chunk_len}, free={free_mem/2**30:.1f}GiB)")
                        else:
                            print(f"[compress] streams (CPU)={gpu_streams} (chunk_len={chunk_len})")
                    except Exception:
                        print(f"[compress] streams={gpu_streams}")

                # Online head adaptation: adapt the prediction head between batches
                adapter = None
                if _HAS_ADAPT and adapt_config is not None:
                    _lr, _K = adapt_config
                    adapter = HeadAdapter(self.model, lr=_lr, adapt_steps=_K)
                    if progress:
                        print(f"[compress] Online head adaptation enabled (lr={_lr}, K={_K})")

                # Process chunks in GPU batches to reduce H2D overhead
                for batch_start in range(0, n_chunks, gpu_streams):
                    batch_end = min(batch_start + gpu_streams, n_chunks)
                    x_list = []
                    Ls_batch = []
                    for i in range(batch_start, batch_end):
                        s = i * chunk_len
                        e = min(s + chunk_len, total_size)
                        sl = mm[s:e]
                        # Torch tensor on CPU; GPU transfer handled inside compress() call
                        t = torch.from_numpy(np.ascontiguousarray(sl)).unsqueeze(0)
                        x_list.append(t)
                        Ls_batch.append(int(e - s))

                    compressed_list, fb_batch, _Ls = compress(
                        self.model, x_list, device=device, progress=progress
                    )
                    # Stream write compressed payload and record offsets/lengths
                    for j, comp_u32 in enumerate(compressed_list):
                        # Ensure deterministic little-endian serialization of u32 words
                        comp_arr = np.asarray(comp_u32, dtype=np.uint32)
                        if comp_arr.dtype.byteorder == '>':
                            comp_arr = comp_arr.byteswap().newbyteorder('<')
                        comp_bytes = comp_arr.tobytes(order='C')
                        f_out.write(comp_bytes)
                        offsets.append(off)
                        lengths.append(len(comp_bytes))
                        off += len(comp_bytes)
                    first_bytes.extend(int(b) & 0xFF for b in fb_batch)

                    # Adapt head on the batch we just compressed
                    if adapter is not None:
                        adapter.adapt_on_batch(x_list, device=device)

                # Build and write index (IDX2: only lengths, no offsets)
                idx = bytearray()
                idx += self.IDX
                idx += bytes(first_bytes)  # n bytes
                for L in lengths:
                    idx += _uvarint_encode(L)
                crc = zlib.crc32(idx) & 0xFFFFFFFF
                f_out.write(idx)
                f_out.write(struct.pack('<I', crc))

                # Restore head to original state so decompress starts identically
                if adapter is not None:
                    adapter.restore_head()

            # Update object state (do not keep compressed payload in RAM)
            self.compressed_data = []
            self.first_bytes = first_bytes
            self.lengths = [chunk_len] * (n_chunks - 1) + [last_chunk_len]
            print(f"Compression complete: {n_chunks} chunks, chunk_len={chunk_len}, last={last_chunk_len}")

            # Energy tracking: stop and report
            if _tracker is not None:
                _tracker.stop()
                print(f"[energy] Compression energy: {_tracker.summary_str()}")
                csv_path = str(self.filepath) + ".compress_energy.csv"
                _tracker.save_csv(csv_path)
                print(f"[energy] Saved to {csv_path}")

        def read_from_disk(self):
            self._read_file()
            print("File loaded successfully")

        def decompress(self, progress: bool = True) -> bytes:
            self._read_file()
            print(f"Total compressed size from disk: {sum(len(c) for c in self.compressed_data)} bytes")

            # Energy tracking
            _tracker = None
            if _energy_tracker is not None:
                _tracker = _energy_tracker()
                _tracker.start()
                if progress:
                    print("[energy] Started energy measurement for decompression")

            # Decompress in batches to limit GPU memory and align with encoder batch semantics
            try:
                gpu_streams = int(os.getenv("BOA_GPU_STREAMS", "5000"))
            except Exception:
                gpu_streams = 5000
            n = len(self.compressed_data)
            gpu_streams = max(1, min(int(gpu_streams), int(n)))
            if progress:
                try:
                    if device == "cuda" and torch.cuda.is_available():
                        free_mem, total_mem = torch.cuda.mem_get_info()
                        print(f"[decompress] gpu_streams={gpu_streams} (free={free_mem/2**30:.1f}GiB)")
                    else:
                        print(f"[decompress] streams (CPU)={gpu_streams}")
                except Exception:
                    print(f"[decompress] streams={gpu_streams}")

            # Online head adaptation for decompression (must mirror encoder)
            adapter = None
            if _HAS_ADAPT and adapt_config is not None:
                _lr, _K = adapt_config
                adapter = HeadAdapter(self.model, lr=_lr, adapt_steps=_K)
                if progress:
                    print(f"[decompress] Online head adaptation enabled (lr={_lr}, K={_K})")

            out_parts: list[bytes] = []
            for batch_start in range(0, n, gpu_streams):
                batch_end = min(batch_start + gpu_streams, n)
                comp_u32_batch = [np.frombuffer(c, dtype='<u4').copy() for c in self.compressed_data[batch_start:batch_end]]
                lens_batch = self.lengths[batch_start:batch_end]
                fb_batch = self.first_bytes[batch_start:batch_end]

                decoded_list = decompress(
                    self.model, comp_u32_batch, lens_batch, fb_batch, device=device, progress=progress
                )
                # Adapt head on the batch we just decompressed (mirrors encoder)
                if adapter is not None:
                    byte_seqs = [torch.from_numpy(d.flatten()).unsqueeze(0) for d in decoded_list]
                    adapter.adapt_on_batch(byte_seqs, device=device)

                for d in decoded_list:
                    out_parts.append(d.tobytes() if hasattr(d, "tobytes") else bytes(d))

            # Restore head to original state (clean up after decompression)
            if adapter is not None:
                adapter.restore_head()

            # Energy tracking: stop and report
            if _tracker is not None:
                _tracker.stop()
                print(f"[energy] Decompression energy: {_tracker.summary_str()}")
                csv_path = str(self.filepath) + ".decompress_energy.csv"
                _tracker.save_csv(csv_path)
                print(f"[energy] Saved to {csv_path}")

            return b"".join(out_parts)

        def get_metadata(self):
            return dict(self.metadata)

    return BoaFile(filepath, model)


Writing boa.py


**`train.py`** — training & evaluation loop.


In [10]:
%%writefile train.py
import torch
import torch.nn as nn
from datetime import datetime
from tqdm.auto import tqdm
import numpy as np
import time

# Optional wandb import — gracefully degrade if not installed
try:
    import wandb as _wandb
    _HAS_WANDB = True
except ImportError:
    _wandb = None
    _HAS_WANDB = False

# --- eval loop (reports mean bpp across the loader) ---
@torch.inference_mode()
def evaluate_bpp(model, loader, criterion, device="cuda", vocab_size=256):
    model.eval().to(device)
    total_loss = 0.0
    total_tokens = 0
    for batch in loader:
        x = batch[:, :-1].to(device)
        y = batch[:, 1:].to(device)
        logits = model(x)  # [B, L-1, vocab_size]
        loss = criterion(logits.reshape(-1, vocab_size), y.reshape(-1))
        total_loss += loss.item() * y.numel()
        total_tokens += y.numel()
    mean_nll = total_loss / max(1, total_tokens)
    bpp = mean_nll / np.log(2)  # bits per input byte
    return bpp

def train(model, train_loader, val_loader, test_loader, optimizer, criterion, device="cuda", name="BoaBytePredictor", NUM_EPOCHS=10, PRECISION="fp32", progress=True, start_epoch=1, vocab_size=256, patience=0, scheduler=None, grad_clip=0.0, position_weights=None, wandb_run=None):

    IS_CUDA = torch.cuda.is_available() and device == "cuda"

    # Pre-process position weights (if provided) for fast per-token weighting
    _pw = None
    if position_weights is not None:
        # position_weights: [seq_len] float tensor — weight for each target position
        _pw = torch.tensor(position_weights, dtype=torch.float32, device=device)
        print(f"[INFO] Position-weighted loss: coarse={position_weights[0]:.2f}, fine={position_weights[-1]:.2f}")

    print(f"[INFO] Using precision = {PRECISION}")
    if patience > 0:
        print(f"[INFO] Early stopping enabled: patience = {patience} epochs")
    if scheduler is not None:
        print(f"[INFO] LR scheduler: {scheduler.__class__.__name__}")
    if grad_clip > 0:
        print(f"[INFO] Gradient clipping: max_norm={grad_clip}")

    def get_autocast_dtype(precision):
        if precision == "bf16":
            return torch.bfloat16
        elif precision == "fp16":
            return torch.float16
        elif precision == "fp8":
            try:
                return torch.float8_e5m2  # Hopper architecture only (H100 / RTX 5090)
            except AttributeError:
                print("[WARN] FP8 not supported on this PyTorch build, falling back to FP16")
                return torch.float16
        else:
            return torch.float32

    AUTODTYPE = get_autocast_dtype(PRECISION)
    amp_enabled = PRECISION in ["bf16", "fp16", "fp8"] and IS_CUDA
    save_half = PRECISION in ["fp16", "fp8"]  # save weights as fp16 when training in reduced precision (bf16 saves as fp32)
    # GradScaler is not needed/used for bf16 (no loss scaling required)
    use_scaler = PRECISION in ["fp16", "fp8"] and IS_CUDA
    scaler = torch.amp.GradScaler("cuda", enabled=use_scaler)

    best_val_bpp = float("inf")
    best_epoch = 0
    epochs_without_improvement = 0
    best_state_dict = None

    # Wandb logging helper
    _wb = wandb_run  # None when wandb is disabled

    # Helper to strip torch.compile's _orig_mod. prefix from state_dict keys
    def _clean_state_dict(sd):
        return {k.replace('_orig_mod.', ''): v for k, v in sd.items()}

    def _get_backbone_name(m):
        # torch.compile wraps model in OptimizedModule; unwrap to find _backbone_name
        inner = getattr(m, '_orig_mod', m)
        return getattr(inner, '_backbone_name', 'unknown')

    model.train().to(device)
    train_steps_per_epoch = len(train_loader)
    total_train_steps = max(1, train_steps_per_epoch)
    global_step = (start_epoch - 1) * total_train_steps
    for epoch in range(start_epoch, NUM_EPOCHS + 1):
        epoch_start = time.perf_counter()
        epoch_tokens = 0
        loader = tqdm(train_loader, total=total_train_steps, desc=f"Epoch {epoch} [{PRECISION}]", disable=not progress)
        for batch in loader:
            x = batch[:, :-1].to(device, non_blocking=True)
            y = batch[:, 1:].to(device, non_blocking=True)
            epoch_tokens += y.numel()

            optimizer.zero_grad(set_to_none=True)

            # --- Automatic mixed precision block ---
            with torch.autocast(device_type=device, dtype=AUTODTYPE, enabled=amp_enabled):
                logits = model(x)
                if _pw is not None:
                    # Per-position weighted loss: tile weights across the sequence
                    # x has shape [B, seq_len-1]; positions repeat with period event_bytes
                    B, L = y.shape
                    pw_len = len(_pw)
                    # Tile position weights to cover L positions
                    reps = (L + pw_len - 1) // pw_len
                    weights = _pw.repeat(reps)[:L]  # [L]
                    weights = weights.unsqueeze(0).expand(B, -1)  # [B, L]
                    import torch.nn.functional as F_local
                    loss_tok = F_local.cross_entropy(
                        logits.reshape(-1, vocab_size), y.reshape(-1), reduction="none"
                    ).reshape(B, L)
                    loss = (loss_tok * weights).sum() / weights.sum()
                else:
                    loss = criterion(logits.reshape(-1, vocab_size), y.reshape(-1))

            # --- Scaled backward for FP16/FP8 ---
            scaler.scale(loss).backward()
            if grad_clip > 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip)
            scaler.step(optimizer)
            scaler.update()

            # --- Progress ---
            bits_per_byte = loss.item() / np.log(2)
            global_step += 1
            if progress:
                loader.set_postfix(
                    loss=f"{loss.item():.4f}",
                    bits=f"{bits_per_byte:.3f}",
                    ratio=f"{(8 / bits_per_byte):.2f}x"
                )

            # --- Wandb step-level logging ---
            if _wb is not None:
                _wb.log({
                    "train/loss": loss.item(),
                    "train/bpp": bits_per_byte,
                    "train/compression_ratio": 8.0 / max(bits_per_byte, 1e-8),
                }, step=global_step)

        epoch_elapsed = time.perf_counter() - epoch_start
        tok_per_sec = epoch_tokens / max(epoch_elapsed, 1e-6)
        mb_per_sec = epoch_tokens / (1024 * 1024) / max(epoch_elapsed, 1e-6)
        print(f"  Epoch {epoch} throughput: {tok_per_sec:,.0f} tok/s ({mb_per_sec:.2f} MB/s), {epoch_elapsed:.1f}s")

        _sd = _clean_state_dict(model.state_dict())
        if save_half:
            _sd = {k: v.half() if v.is_floating_point() else v for k, v in _sd.items()}
        _ckpt = {
            'state_dict': _sd,
            'backbone': _get_backbone_name(model),
        }
        torch.save(_ckpt, f"{name}_{datetime.now().strftime('%dth%b')}_Checkpoint_epoch_{epoch}_{PRECISION}.pt")
        val_bpp = evaluate_bpp(model, val_loader, criterion, device=device, vocab_size=vocab_size)
        current_lr = optimizer.param_groups[0]['lr']
        print(f"[Epoch {epoch}] val bpp={val_bpp:.4f} (ratio ~ {8/val_bpp:.2f}x)  lr={current_lr:.2e}")

        # --- Wandb epoch-level logging ---
        if _wb is not None:
            _wb.log({
                "epoch": epoch,
                "val/bpp": val_bpp,
                "val/compression_ratio": 8.0 / max(val_bpp, 1e-8),
                "train/throughput_tok_s": tok_per_sec,
                "train/throughput_MB_s": mb_per_sec,
                "train/epoch_time_s": epoch_elapsed,
                "train/lr": current_lr,
                "best/val_bpp": best_val_bpp,
                "best/epoch": best_epoch,
            }, step=global_step)

        if scheduler is not None:
            scheduler.step()

        # --- Early stopping logic ---
        if val_bpp < best_val_bpp:
            best_val_bpp = val_bpp
            best_epoch = epoch
            epochs_without_improvement = 0
            import copy
            best_state_dict = copy.deepcopy(model.state_dict())
        else:
            epochs_without_improvement += 1
            if patience > 0:
                print(f"  [early stop] no improvement for {epochs_without_improvement}/{patience} epochs (best={best_val_bpp:.4f} @ epoch {best_epoch})")
            if patience > 0 and epochs_without_improvement >= patience:
                print(f"  [early stop] Stopping at epoch {epoch}. Restoring best model from epoch {best_epoch}.")
                break

    # Restore best model if early stopping was used
    if best_state_dict is not None and patience > 0:
        model.load_state_dict(best_state_dict)
        print(f"  [early stop] Restored best model (val bpp={best_val_bpp:.4f}, epoch {best_epoch})")

    _final_sd = _clean_state_dict(model.state_dict())
    if save_half:
        _final_sd = {k: v.half() if v.is_floating_point() else v for k, v in _final_sd.items()}
    _final_ckpt = {
        'state_dict': _final_sd,
        'backbone': _get_backbone_name(model),
    }
    torch.save(_final_ckpt, f"{name}_final_model_{PRECISION}.pt")
    test_bpp = evaluate_bpp(model, test_loader, criterion, device=device, vocab_size=vocab_size)
    print(f"[TEST] bpp={test_bpp:.4f}  ratio ~ {8/test_bpp:.2f}x")

    # --- Wandb final summary ---
    if _wb is not None:
        _wb.log({
            "test/bpp": test_bpp,
            "test/compression_ratio": 8.0 / max(test_bpp, 1e-8),
        }, step=global_step)
        _wb.summary["test_bpp"] = test_bpp
        _wb.summary["test_compression_ratio"] = 8.0 / max(test_bpp, 1e-8)
        _wb.summary["best_val_bpp"] = best_val_bpp
        _wb.summary["best_epoch"] = best_epoch
        _wb.summary["total_epochs"] = epoch


Writing train.py


**`online_adapt.py`** — optional online head adaptation.


In [11]:
%%writefile online_adapt.py
"""
Online head adaptation for BOA compression.

During compression/decompression, the prediction head is continuously
fine-tuned on the bytes being processed.  Both encoder and decoder
see the same bytes and perform identical gradient updates, keeping
their models perfectly synchronized.

This is the neural equivalent of cmix's online model adaptation —
the single most impactful technique in compression.

Design:
  - Backbone (Mamba/LSTM/etc.) stays FROZEN
  - Only the head (Linear → SiLU/ReLU → Linear, ~50K params for d=128)
    is adapted
  - Adaptation happens BETWEEN batches (not inside the streaming loop)
    so we don't fight @torch.inference_mode or Mamba's InferenceParams
  - After each batch of N streams finishes, we take the raw bytes that
    were just compressed/decompressed, run ONE forward pass through the
    full model, and do K gradient steps on the head only
  - Both encoder and decoder process the same bytes → identical updates

Usage in boa.py (between batch loops):
    adapter = HeadAdapter(model, lr=2e-4)
    for batch_start in range(0, n_chunks, gpu_streams):
        # ... compress batch ...
        adapter.adapt_on_batch(batch_bytes, device=device)
"""

import torch
import torch.nn.functional as F
import os

# Environment variable to enable/disable online adaptation
_ENABLE_ADAPT = os.environ.get("BOA_ADAPT", "1") != "0"


class HeadAdapter:
    """Manages online adaptation of the prediction head during compression.

    Between batches, takes the compressed/decompressed bytes and runs
    K gradient steps on the head using the full model forward pass.
    The backbone stays frozen — only head parameters receive gradients.
    """

    def __init__(self, model, lr: float = 2e-4, adapt_steps: int = 1,
                 max_grad_norm: float = 1.0, max_seq_len: int = 4096):
        self.model = model
        self.lr = lr
        self.adapt_steps = adapt_steps
        self.max_grad_norm = max_grad_norm
        self.max_seq_len = max_seq_len  # cap sequence length for adaptation to save memory

        # Identify head parameters
        self.head = model.head
        self.head_params = list(self.head.parameters())

        # Save original head state so we can restore after compress/decompress
        # (ensures encoder and decoder both start from the same checkpoint)
        self._original_state = {k: v.clone() for k, v in self.head.state_dict().items()}

        # Freeze everything except head
        for param in model.parameters():
            param.requires_grad_(False)
        for param in self.head_params:
            param.requires_grad_(True)

        # Deterministic SGD — no momentum so encoder/decoder produce
        # bitwise identical updates given the same data
        self.optimizer = torch.optim.SGD(self.head_params, lr=lr)

    def restore_head(self):
        """Restore head to original (pre-adaptation) state.
        Call after compress() finishes so that decompress() starts identically.
        """
        self.head.load_state_dict(self._original_state)
        self._de_inference_head()
        # Reset optimizer state
        self.optimizer = torch.optim.SGD(self.head_params, self.lr)

    def _de_inference_head(self):
        """Replace any inference-mode-tainted parameters with fresh copies.

        compress_GPU / decompress_GPU use @torch.inference_mode(), which
        permanently marks parameters used in that context as inference
        tensors.  Such tensors cannot participate in autograd.
        We fix this by replacing them with non-inference clones.
        """
        for name, module in self.head.named_modules():
            if isinstance(module, torch.nn.Linear):
                if module.weight.is_inference():
                    module.weight = torch.nn.Parameter(
                        module.weight.data.clone(), requires_grad=True
                    )
                if module.bias is not None and module.bias.is_inference():
                    module.bias = torch.nn.Parameter(
                        module.bias.data.clone(), requires_grad=True
                    )
        # Refresh param list and optimizer after parameter replacement
        self.head_params = list(self.head.parameters())
        for p in self.head_params:
            p.requires_grad_(True)
        self.optimizer = torch.optim.SGD(self.head_params, self.lr)

    def _get_backbone_features(self, x: torch.Tensor) -> torch.Tensor:
        """Run backbone only (frozen, no grad) to get hidden states.

        x: [B, L] long tensor of byte sequences
        Returns: [B, L, D] plain tensor (no grad, no inference mode)
        """
        # Use no_grad + explicit inference_mode(False) to ensure we get
        # plain tensors even if Mamba blocks have cached inference tensors
        with torch.no_grad(), torch.inference_mode(False):
            h = self.model.embedding(x)  # [B, L, D]

            # Check if model has inference_params-style forward
            if hasattr(self.model, '_backbone_name') and \
               self.model._backbone_name in ('mamba', 'mambav1', 'mamba2'):
                # For Mamba models, use the full forward pass through blocks
                # (NOT streaming mode — full sequence parallel)
                for blk in self.model.blocks:
                    h = blk(h)  # No inference_params → runs in training/parallel mode
            else:
                for blk in self.model.blocks:
                    h = blk(h)

            # Apply final_norm if present
            if hasattr(self.model, 'final_norm'):
                h = self.model.final_norm(h)

        return h.detach().clone()  # clone ensures plain tensor, no inference mode residue

    def adapt_on_batch(self, byte_sequences: list, device: str = "cuda"):
        """Adapt the head on a batch of byte sequences.

        byte_sequences: list of 1D byte tensors or numpy arrays (the raw
                        bytes that were just compressed/decompressed).
        device: torch device string.

        Must be called OUTSIDE @torch.inference_mode() — we explicitly
        disable inference mode and fix tainted parameters here.
        """
        if not byte_sequences or self.adapt_steps <= 0:
            return

        # Fix parameters tainted by @torch.inference_mode() in codec
        self._de_inference_head()

        # Explicitly exit inference mode for the adaptation step
        with torch.inference_mode(False):
            self._adapt_on_batch_impl(byte_sequences, device)

    def _adapt_on_batch_impl(self, byte_sequences: list, device: str):
        """Inner implementation of adapt_on_batch (runs outside inference mode)."""
        # Convert to tensor and truncate to max_seq_len for memory efficiency
        def _seq_len(s):
            if isinstance(s, torch.Tensor):
                return s.numel()
            return len(s) if hasattr(s, '__len__') else int(s.shape[-1])
        max_len = min(max(_seq_len(s) for s in byte_sequences), self.max_seq_len)
        batch_size = len(byte_sequences)

        # Build padded tensor [B, L]
        x = torch.zeros(batch_size, max_len, dtype=torch.long, device=device)
        for i, seq in enumerate(byte_sequences):
            if isinstance(seq, torch.Tensor):
                s = seq.to(device).flatten()[:max_len]
            elif isinstance(seq, (bytes, bytearray)):
                s = torch.tensor(list(seq[:max_len]), dtype=torch.long, device=device)
            else:
                import numpy as np
                s = torch.from_numpy(np.frombuffer(seq, dtype=np.uint8)[:max_len].copy()).to(device)
            x[i, :len(s)] = s.to(torch.long)

        # Input: x[:, :-1], Target: x[:, 1:]
        x_in = x[:, :-1]
        y = x[:, 1:]

        if x_in.shape[1] < 1:
            return

        # Get backbone features (frozen, no grad)
        h = self._get_backbone_features(x_in)  # [B, L-1, D]

        # Adaptation steps: gradient only flows through the head
        self.head.train()
        for _ in range(self.adapt_steps):
            logits = self.head(h)  # [B, L-1, V]
            V = logits.shape[-1]
            loss = F.cross_entropy(logits.reshape(-1, V), y.reshape(-1))

            self.optimizer.zero_grad()
            loss.backward()
            if self.max_grad_norm > 0:
                torch.nn.utils.clip_grad_norm_(self.head_params, self.max_grad_norm)
            self.optimizer.step()

        # Back to eval for inference
        self.head.eval()

    def adapt_on_raw_bytes(self, raw_bytes: bytes, chunk_len: int,
                           device: str = "cuda", max_chunks: int = 64):
        """Convenience: adapt on raw bytes split into chunks.

        Takes a contiguous byte buffer and splits it into chunks for adaptation.
        Useful when called from boa.py.
        """
        import numpy as np
        total = len(raw_bytes)
        n_chunks = min((total + chunk_len - 1) // chunk_len, max_chunks)

        seqs = []
        for i in range(n_chunks):
            s = i * chunk_len
            e = min(s + chunk_len, total)
            seqs.append(torch.tensor(list(raw_bytes[s:e]), dtype=torch.long))

        self.adapt_on_batch(seqs, device=device)


Writing online_adapt.py


## 5 · Build the model

`BoaConstrictor`'s factory chooses `mamba_ssm` when it is built with `device="cuda"`. To
honour *"use `mamba_ssm` where possible, else `mambapy`"* we try the CUDA factory first and,
if the fused kernels are missing, build the **same architecture** with the pure-torch
`mambapy` path (factory `device="cpu"`) and move it onto the GPU. Non-Mamba backbones
(`transformer`, `gru`, …) are pure-torch and need neither package.


In [12]:
import numpy as np, torch
from model import BoaConstrictor, ByteDataloader, make_splits

torch.manual_seed(CONFIG["seed"]); np.random.seed(CONFIG["seed"])
if DEVICE == "cuda":
    torch.cuda.manual_seed_all(CONFIG["seed"])

def build_model(cfg, device, have_mamba_ssm):
    bb = cfg["backbone"]
    is_mamba = bb in ("mamba", "mambav1", "mamba2")
    kw = dict(d_model=cfg["d_model"], num_layers=cfg["num_layers"], vocab_size=cfg["vocab_size"])
    # returns (model, runs_on_cpu): runs_on_cpu=False only for the fused mamba_ssm
    # (CUDA-only) path, so the compression step knows whether the CPU codec is an option.
    if is_mamba and device == "cuda" and have_mamba_ssm:
        try:
            m = BoaConstrictor(**kw, device="cuda", backbone=bb)
            print(f"[model] {bb} via mamba_ssm (fused CUDA)")
            return m.to(device), False
        except Exception as e:
            print(f"[model] mamba_ssm build failed ({e}); using mambapy.")
    if is_mamba:
        # pure-torch Mamba (mambapy): build for 'cpu' then move to the real device
        m = BoaConstrictor(**kw, device="cpu", backbone=bb)
        print(f"[model] {bb} via mambapy (pure-torch) on {device}")
        return m.to(device), True
    m = BoaConstrictor(**kw, device=device, backbone=bb)   # pure-torch non-Mamba backbones
    print(f"[model] {bb} (pure-torch) on {device}")
    return m.to(device), True

model, MODEL_RUNS_ON_CPU = build_model(CONFIG, DEVICE, HAVE_MAMBA_SSM)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")


[model] mambav1 via mamba_ssm (fused CUDA)
Parameters: 169,152


## 6 · Train the byte predictor

We split the bytes into train/val/test, then minimise cross-entropy between the predicted
next-byte distribution and the true next byte. The loop reports **bits per byte (bpp)** — the
expected code length per input byte — and `8/bpp` is the model's predicted compression ratio.
On a CPU-only runtime this is slow; reduce `data_mb`, `seq_len`, and `epochs`.


In [13]:
from train import train

train_b, val_b, test_b = make_splits(data, CONFIG["seq_len"], CONFIG["batch_size"], splits=tuple(CONFIG["splits"]))
mk = lambda b: ByteDataloader(b, seq_len=CONFIG["seq_len"], batch_size=CONFIG["batch_size"], device=DEVICE)
train_loader, val_loader, test_loader = mk(train_b), mk(val_b), mk(test_b)
print(f"train/val/test bytes: {len(train_b):,} / {len(val_b):,} / {len(test_b):,}  "
      f"({len(train_loader)} train batches/epoch)")

optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["lr"], weight_decay=0.01)
criterion = torch.nn.CrossEntropyLoss()

train(model, train_loader, val_loader, test_loader, optimizer, criterion,
      device=DEVICE, name="cms_experiment", NUM_EPOCHS=CONFIG["epochs"],
      PRECISION=CONFIG["precision"], progress=True, vocab_size=CONFIG["vocab_size"])


train/val/test bytes: 39,900,000 / 4,950,000 / 5,050,000  (798 train batches/epoch)
[INFO] Using precision = fp32


Epoch 1 [fp32]:   0%|          | 0/798 [00:00<?, ?it/s]

  Epoch 1 throughput: 877,862 tok/s (0.84 MB/s), 45.4s
[Epoch 1] val bpp=2.3559 (ratio ~ 3.40x)  lr=5.00e-04


Epoch 2 [fp32]:   0%|          | 0/798 [00:00<?, ?it/s]

  Epoch 2 throughput: 904,539 tok/s (0.86 MB/s), 44.1s
[Epoch 2] val bpp=2.2620 (ratio ~ 3.54x)  lr=5.00e-04


...

  Epoch 6 throughput: 856,197 tok/s (0.82 MB/s), 46.6s
[Epoch 6] val bpp=2.1450 (ratio ~ 3.73x)  lr=5.00e-04


Epoch 7 [fp32]:   0%|          | 0/798 [00:00<?, ?it/s]

  Epoch 7 throughput: 857,303 tok/s (0.82 MB/s), 46.5s
[Epoch 7] val bpp=2.1355 (ratio ~ 3.75x)  lr=5.00e-04


Epoch 8 [fp32]:   0%|          | 0/798 [00:00<?, ?it/s]

  Epoch 8 throughput: 847,514 tok/s (0.81 MB/s), 47.1s
[Epoch 8] val bpp=2.1219 (ratio ~ 3.77x)  lr=5.00e-04
[TEST] bpp=2.1183  ratio ~ 3.78x


## 7 · Compress

`BOA(device, ...)` selects the entropy coder: on CUDA it uses the GPU range coder (built on
first import); otherwise `constriction` on CPU. We first probe the GPU coder and fall back to
CPU coding if its CUDA build/run isn't available. `chunks_count` splits the file into
independently-coded chunks (parallelism + streaming); `None` targets ~2 KB per chunk.


In [14]:
import math, time
from boa import BOA

def select_codec_device(device):
    if device != "cuda":
        return "cpu"
    try:
        import gpu_range_coder as _g          # triggers the nvcc build
        _g.gpu.queue.RangeCoderBatch(1, CONFIG["vocab_size"], 8)  # tiny sanity construct
        print("[codec] GPU range coder ready (CUDA extension built)")
        return "cuda"
    except Exception as e:
        print(f"[codec] GPU range coder unavailable ({e}); using constriction CPU codec")
        return "cpu"

CODEC_DEVICE = select_codec_device(DEVICE)
if CODEC_DEVICE == "cpu" and DEVICE == "cuda" and not MODEL_RUNS_ON_CPU:
    raise RuntimeError(
        "GPU range coder unavailable, but the model uses mamba_ssm (CUDA-only) and cannot "
        "run under the CPU codec. Re-run with CONFIG['try_mamba_ssm']=False (pure-torch "
        "mambapy, which supports both codecs), or enable nvcc so the GPU coder can build.")
chunks_count = CONFIG["chunks_count"] or max(1, round(len(data) / 2048))
BOA_PATH = (("/content/" if os.path.isdir("/content") else "") + "cms_experiment.boa")

boa = BOA(CODEC_DEVICE, BOA_PATH, model)
t0 = time.time()
boa.compress(data_path=DATA_PATH, chunks_count=chunks_count, progress=True)
comp_s = time.time() - t0

comp_size = os.path.getsize(BOA_PATH)
print(f"\nOriginal:   {len(data):,} bytes")
print(f"Compressed: {comp_size:,} bytes  ({chunks_count} chunks)")
print(f"Ratio (excl. model): {len(data)/comp_size:.3f}x   |   "
      f"bits/byte: {8*comp_size/len(data):.3f}")
print(f"Compression time: {comp_s:.1f}s ({len(data)/1e6/max(comp_s,1e-9):.2f} MB/s)")


[codec] GPU range coder ready (CUDA extension built)
[compress] gpu_streams=5000 (chunk_len=2048, free=13.6GiB)


/content/boa.py:280: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:203.)
  t = torch.from_numpy(np.ascontiguousarray(sl)).unsqueeze(0)


Compress (GPU streams x5000):   0%|          | 0.0/9995.1171875 [00:00<?, ?KB/s]

Compress (GPU streams x5000):   0%|          | 0.0/9995.1171875 [00:00<?, ?KB/s]

Compress (GPU streams x5000):   0%|          | 0.0/9995.1171875 [00:00<?, ?KB/s]

Compress (GPU streams x5000):   0%|          | 0.0/9995.1171875 [00:00<?, ?KB/s]

Compress (GPU streams x4375):   0%|          | 0.0/8745.7275390625 [00:00<?, ?KB/s]

Compression complete: 24375 chunks, chunk_len=2048, last=2048

Original:   49,920,000 bytes
Compressed: 13,394,042 bytes  (24375 chunks)
Ratio (excl. model): 3.727x   |   bits/byte: 2.146
Compression time: 34.0s (1.47 MB/s)


## 8 · Decompress & verify the round-trip

Decompression replays the **identical** model predictions to invert the range coder. We
confirm the result is **bit-for-bit identical** to the input (SHA-256), then reshape back to
`N × 24` float32 and check the physics values are recovered exactly (max abs diff = 0).


In [15]:
import hashlib

t0 = time.time()
restored = boa.decompress(progress=True)
dec_s = time.time() - t0

ok = restored == data
h_in, h_out = hashlib.sha256(data).hexdigest(), hashlib.sha256(restored).hexdigest()
print(f"\nDecompression time: {dec_s:.1f}s ({len(restored)/1e6/max(dec_s,1e-9):.2f} MB/s)")
print(f"Original     SHA-256: {h_in}")
print(f"Decompressed SHA-256: {h_out}")
print("✅ LOSSLESS round-trip CONFIRMED" if ok else "❌ MISMATCH")
assert ok, "round-trip failed!"

orig_f = np.frombuffer(data, dtype=np.float32).reshape(-1, CONFIG["n_features"])
deco_f = np.frombuffer(restored, dtype=np.float32).reshape(-1, CONFIG["n_features"])
print(f"Recovered {deco_f.shape[0]:,} jets — max |orig - decompressed| = "
      f"{np.max(np.abs(orig_f - deco_f))}")


Total compressed size from disk: 13320860 bytes
[decompress] gpu_streams=5000 (free=13.3GiB)


Decompress (GPU streams x5000):   0%|          | 0.0/9995.1171875 [00:00<?, ?KB/s]

Decompress (GPU streams x5000):   0%|          | 0.0/9995.1171875 [00:00<?, ?KB/s]

Decompress (GPU streams x5000):   0%|          | 0.0/9995.1171875 [00:00<?, ?KB/s]

Decompress (GPU streams x5000):   0%|          | 0.0/9995.1171875 [00:00<?, ?KB/s]

Decompress (GPU streams x4375):   0%|          | 0.0/8745.7275390625 [00:00<?, ?KB/s]


Decompression time: 32.8s (1.52 MB/s)
Original     SHA-256: edd6e4dd573acf7fb83b352726f5a23062572a5139cd268bd3dd9561d133dec6
Decompressed SHA-256: edd6e4dd573acf7fb83b352726f5a23062572a5139cd268bd3dd9561d133dec6
✅ LOSSLESS round-trip CONFIRMED
Recovered 520,000 jets — max |orig - decompressed| = 0.0


## 9 · (Optional) classical baselines

For context, here is how general-purpose compressors do on the same bytes. Neural
compression typically wins on this structured float data — at the cost of compute and a model
that must be shipped/amortised alongside the file (see the paper for model-inclusive ratios).


In [16]:
import zlib, lzma, bz2
for name, comp in [("zlib -9", lambda d: zlib.compress(d, 9)),
                   ("bz2 -9",  lambda d: bz2.compress(d, 9)),
                   ("lzma -9", lambda d: lzma.compress(d, preset=9 | lzma.PRESET_EXTREME))]:
    cb = comp(data)
    print(f"{name:8s}: {len(d:=cb):>12,} bytes   ratio {len(data)/len(cb):.3f}x")
print(f"{'BOA':8s}: {comp_size:>12,} bytes   ratio {len(data)/comp_size:.3f}x   <-- neural")


zlib -9 :   19,463,909 bytes   ratio 2.565x
bz2 -9  :   18,136,704 bytes   ratio 2.752x
lzma -9 :   15,489,644 bytes   ratio 3.223x
BOA     :   13,394,042 bytes   ratio 3.727x   <-- neural


## Notes & next steps

- **Scale up to the paper config:** `data_mb=None` (full 47.6 MB), `epochs=40`,
  `chunks_count=10000`. Use a stronger backbone/size (e.g. `d_model=256`) for a better ratio.
- **Speed:** the GPU range coder codes chunks in parallel; more, smaller chunks = more
  parallel streams. `mamba_ssm` trains ~5–10× faster than the `mambapy` fallback.
- **Model overhead:** the `.boa` file excludes the model. For a fair end-to-end ratio,
  count the trained weights too (or amortise one model over many files) — see the paper.
- **CPU-only runtimes** work via `constriction` + `mambapy`, just slowly — shrink the config.
- Backbone options: `mamba, mambav1, mamba2, transformer, gru, lstm, mingru, rwkv6/7/8,
  griffin, xlstm`. `mamba2` needs CUDA + `mamba_ssm`.

Citation: A. Gupta, C. Doglioni, T. J. Elliott — *BOA constrictor: a Mamba-based lossless
compressor for scientific data*, Mach. Learn.: Sci. Technol. **7** 035014 (2026),
[doi:10.1088/2632-2153/ae64a9](https://doi.org/10.1088/2632-2153/ae64a9).


In [17]:
try:
    from codecarbon import EmissionsTracker
except ImportError:
    %pip -q install codecarbon
    from codecarbon import EmissionsTracker

import os, time, pandas as pd

os.makedirs("code_carbon", exist_ok=True)

def sync_cuda():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

def state_size_bytes(m, name):
    path = f"{name}_state_dict.pt"
    torch.save(m.state_dict(), path)
    size = os.path.getsize(path)
    os.remove(path)
    return size

def measure_boa_carbon(method, m, boa_path, model_runs_on_cpu):
    if CODEC_DEVICE == "cpu" and DEVICE == "cuda" and not model_runs_on_cpu:
        raise RuntimeError("CPU codec cannot run this CUDA-only Mamba model.")

    csv_name = f"{method}_emissions.csv"
    csv_path = os.path.join("code_carbon", csv_name)

    for path in [boa_path, csv_path]:
        if os.path.exists(path):
            os.remove(path)

    tracker = EmissionsTracker(
        output_dir="./code_carbon/",
        output_file=csv_name,
        log_level="error",
    )

    b = BOA(CODEC_DEVICE, boa_path, m)

    sync_cuda()
    tracker.start()
    t0 = time.time()

    b.compress(DATA_PATH, chunks_count=chunks_count, progress=True)
    restored = b.decompress(progress=True)

    sync_cuda()
    emissions = tracker.stop()
    runtime_s = time.time() - t0

    assert restored == data, f"{method}: round-trip failed"

    row = pd.read_csv(csv_path).iloc[-1]
    energy_col = "energy_consumed" if "energy_consumed" in row else "energy_kWh"

    boa_bytes = os.path.getsize(boa_path)
    model_bytes = state_size_bytes(m, method)
    input_mb = len(data) / 1e6
    saved_mb = (len(data) - boa_bytes) / 1e6

    return {
        "method": method,
        "actual_bits_per_byte": 8 * boa_bytes / len(data),
        "ratio_excl_model": len(data) / boa_bytes,
        "ratio_incl_model": len(data) / (boa_bytes + model_bytes),
        "runtime_s_codecarbon": float(row.get("duration", runtime_s)),
        "energy_kWh": float(row[energy_col]),
        "emissions_kgCO2eq": float(emissions if emissions is not None else row["emissions"]),
        "kgCO2eq_per_input_MB": float(row["emissions"]) / input_mb,
        "kgCO2eq_per_saved_MB": float(row["emissions"]) / saved_mb if saved_mb > 0 else float("nan"),
        "compressed_size_bytes": boa_bytes,
        "model_size_bytes": model_bytes,
        "roundtrip_ok": True,
    }

baseline_carbon = measure_boa_carbon(
    "baseline_boa",
    model,
    "baseline_boa_codecarbon.boa",
    MODEL_RUNS_ON_CPU,
)

display(pd.DataFrame([baseline_carbon]))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 384.6/384.6 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 103.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 111.2 MB/s eta 0:00:00


[codecarbon WARNING @ 11:15:16] Multiple instances of codecarbon are allowed to run at the same time.


[compress] gpu_streams=5000 (chunk_len=2048, free=13.3GiB)


Compress (GPU streams x5000):   0%|          | 0.0/9995.1171875 [00:00<?, ?KB/s]

Compress (GPU streams x5000):   0%|          | 0.0/9995.1171875 [00:00<?, ?KB/s]

...

Decompress (GPU streams x5000):   0%|          | 0.0/9995.1171875 [00:00<?, ?KB/s]

Decompress (GPU streams x5000):   0%|          | 0.0/9995.1171875 [00:00<?, ?KB/s]

Decompress (GPU streams x5000):   0%|          | 0.0/9995.1171875 [00:00<?, ?KB/s]

Decompress (GPU streams x4375):   0%|          | 0.0/8745.7275390625 [00:00<?, ?KB/s]

,method,actual_bits_per_byte,ratio_excl_model,ratio_incl_model,runtime_s_codecarbon,energy_kWh,emissions_kgCO2eq,kgCO2eq_per_input_MB,kgCO2eq_per_saved_MB,compressed_size_bytes,model_size_bytes,roundtrip_ok
0,baseline_boa,2.146481,3.72703,3.544087,65.5371,0.001414,0.000196,0.000004,0.000005,13394042,691390,True


In [18]:
import gc
import torch.nn as nn

class PositionAwareBoa(nn.Module):
    def __init__(self, base, n_features):
        super().__init__()
        self.base = base
        self.row_bytes = 4 * n_features
        d = base.embedding.embedding_dim
        self.pos = nn.ModuleList([
            nn.Embedding(self.row_bytes, d),
            nn.Embedding(n_features, d),
            nn.Embedding(4, d),
            nn.Embedding(self.row_bytes, d),
            nn.Embedding(n_features, d),
            nn.Embedding(4, d),
        ])
        self._backbone_name = getattr(base, "_backbone_name", "position_aware")

    @property
    def embedding(self):
        return self.base.embedding

    @property
    def blocks(self):
        return self.base.blocks

    @property
    def head(self):
        return self.base.head

    def pe(self, p, target=False):
        p = p % self.row_bytes
        i = 3 * int(target)
        return self.pos[i](p) + self.pos[i + 1](p // 4) + self.pos[i + 2](p % 4)

    def norm_head(self, h):
        if hasattr(self.base, "final_norm"):
            h = self.base.final_norm(h)
        return self.base.head(h)

    def forward(self, x, inference_params=None):
        p = torch.arange(x.size(1), device=x.device)
        h = self.base.embedding(x) + self.pe(p).unsqueeze(0)
        for block in self.base.blocks:
            h = block(h, inference_params=inference_params)
        return self.norm_head(h + self.pe(p + 1, target=True).unsqueeze(0))

    @torch.inference_mode()
    def init_stream(self, max_len, batch_size=1, device=None, dtype=None):
        return {
            "base": self.base.init_stream(max_len, batch_size, device, dtype),
            "pos": 0,
        }

    @torch.inference_mode()
    def step(self, byte_t, state):
        s = state["base"]
        p = torch.full_like(byte_t, state["pos"])
        h = self.base.embedding(byte_t) + self.pe(p)

        if isinstance(s, list):
            for i, block in enumerate(self.base.blocks):
                h, s[i] = block.step(h, s[i])
        else:
            h = h[:, None]
            for block in self.base.blocks:
                h = block(h, inference_params=s)
            h = h[:, 0]
            if hasattr(s, "seqlen_offset"):
                s.seqlen_offset += 1
            else:
                s.sequence_length_offset = getattr(s, "sequence_length_offset", 0) + 1

        state["pos"] += 1
        return self.norm_head(h + self.pe(p + 1, target=True))

torch.manual_seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])
if DEVICE == "cuda":
    torch.cuda.manual_seed_all(CONFIG["seed"])

position_base, POSITION_AWARE_RUNS_ON_CPU = build_model(CONFIG, DEVICE, HAVE_MAMBA_SSM)
position_model = PositionAwareBoa(position_base, CONFIG["n_features"]).to(DEVICE)

optimizer = torch.optim.AdamW(position_model.parameters(), lr=CONFIG["lr"], weight_decay=0.01)

t0 = time.time()
train(
    position_model,
    train_loader,
    val_loader,
    test_loader,
    optimizer,
    criterion,
    device=DEVICE,
    name="position_aware_boa",
    NUM_EPOCHS=CONFIG["epochs"],
    PRECISION=CONFIG["precision"],
    progress=True,
    vocab_size=CONFIG["vocab_size"],
)
position_train_s = time.time() - t0

gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()

print(f"Position-aware model trained in {position_train_s:.1f}s")

[model] mambav1 via mamba_ssm (fused CUDA)
[INFO] Using precision = fp32


Epoch 1 [fp32]:   0%|          | 0/798 [00:00<?, ?it/s]

  Epoch 1 throughput: 802,672 tok/s (0.77 MB/s), 49.7s
[Epoch 1] val bpp=2.2949 (ratio ~ 3.49x)  lr=5.00e-04


Epoch 2 [fp32]:   0%|          | 0/798 [00:00<?, ?it/s]

  Epoch 2 throughput: 826,044 tok/s (0.79 MB/s), 48.3s
[Epoch 2] val bpp=2.2117 (ratio ~ 3.62x)  lr=5.00e-04


...

  Epoch 6 throughput: 823,468 tok/s (0.79 MB/s), 48.4s
[Epoch 6] val bpp=2.1039 (ratio ~ 3.80x)  lr=5.00e-04


Epoch 7 [fp32]:   0%|          | 0/798 [00:00<?, ?it/s]

  Epoch 7 throughput: 825,308 tok/s (0.79 MB/s), 48.3s
[Epoch 7] val bpp=2.0957 (ratio ~ 3.82x)  lr=5.00e-04


Epoch 8 [fp32]:   0%|          | 0/798 [00:00<?, ?it/s]

  Epoch 8 throughput: 826,785 tok/s (0.79 MB/s), 48.3s
[Epoch 8] val bpp=2.0899 (ratio ~ 3.83x)  lr=5.00e-04
[TEST] bpp=2.0863  ratio ~ 3.83x
Position-aware model trained in 403.5s


In [19]:
position_carbon = measure_boa_carbon(
    "position_aware_boa",
    position_model,
    "position_aware_boa_codecarbon.boa",
    POSITION_AWARE_RUNS_ON_CPU,
)

carbon_comparison = pd.DataFrame([baseline_carbon, position_carbon])

carbon_comparison["delta_emissions_vs_baseline"] = (
    carbon_comparison["emissions_kgCO2eq"] - baseline_carbon["emissions_kgCO2eq"]
)

carbon_comparison["delta_ratio_excl_model_vs_baseline"] = (
    carbon_comparison["ratio_excl_model"] - baseline_carbon["ratio_excl_model"]
)

carbon_comparison["delta_ratio_incl_model_vs_baseline"] = (
    carbon_comparison["ratio_incl_model"] - baseline_carbon["ratio_incl_model"]
)

carbon_comparison.to_csv("boa_position_aware_carbon_comparison.csv", index=False)

display(carbon_comparison.style.format({
    "actual_bits_per_byte": "{:.4f}",
    "ratio_excl_model": "{:.3f}",
    "ratio_incl_model": "{:.3f}",
    "runtime_s_codecarbon": "{:.1f}",
    "energy_kWh": "{:.6e}",
    "emissions_kgCO2eq": "{:.6e}",
    "kgCO2eq_per_input_MB": "{:.6e}",
    "kgCO2eq_per_saved_MB": "{:.6e}",
    "delta_emissions_vs_baseline": "{:.6e}",
    "delta_ratio_excl_model_vs_baseline": "{:.3f}",
    "delta_ratio_incl_model_vs_baseline": "{:.3f}",
}).hide(axis="index"))

best = carbon_comparison.loc[carbon_comparison["kgCO2eq_per_saved_MB"].idxmin(), "method"]
print(f"Carbon efficiency winner: {best}")
print("saved: boa_position_aware_carbon_comparison.csv")

[compress] gpu_streams=5000 (chunk_len=2048, free=14.3GiB)


Compress (GPU streams x5000):   0%|          | 0.0/9995.1171875 [00:00<?, ?KB/s]

Compress (GPU streams x5000):   0%|          | 0.0/9995.1171875 [00:00<?, ?KB/s]

Compress (GPU streams x5000):   0%|          | 0.0/9995.1171875 [00:00<?, ?KB/s]

Compress (GPU streams x5000):   0%|          | 0.0/9995.1171875 [00:00<?, ?KB/s]

...

Decompress (GPU streams x5000):   0%|          | 0.0/9995.1171875 [00:00<?, ?KB/s]

Decompress (GPU streams x5000):   0%|          | 0.0/9995.1171875 [00:00<?, ?KB/s]

Decompress (GPU streams x4375):   0%|          | 0.0/8745.7275390625 [00:00<?, ?KB/s]

method,actual_bits_per_byte,ratio_excl_model,ratio_incl_model,runtime_s_codecarbon,energy_kWh,emissions_kgCO2eq,kgCO2eq_per_input_MB,kgCO2eq_per_saved_MB,compressed_size_bytes,model_size_bytes,roundtrip_ok,delta_emissions_vs_baseline,delta_ratio_excl_model_vs_baseline,delta_ratio_incl_model_vs_baseline
baseline_boa,2.1465,3.727,3.544,65.5,1.414280e-03,1.962311e-04,3.930912e-06,5.372375e-06,13394042,691390,True,0.000000e+00,0.000,0.000
position_aware_boa,2.1110,3.790,3.584,80.4,1.708444e-03,2.370463e-04,4.748525e-06,6.450666e-06,13172422,757694,True,4.081521e-05,0.063,0.040


Carbon efficiency winner: baseline_boa
saved: boa_position_aware_carbon_comparison.csv


In [20]:
e2e_csv = "baseline_boa_end_to_end_emissions.csv"
e2e_csv_path = os.path.join("code_carbon", e2e_csv)

for path in [e2e_csv_path, "baseline_boa_end_to_end.boa"]:
    if os.path.exists(path):
        os.remove(path)

torch.manual_seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])
if DEVICE == "cuda":
    torch.cuda.manual_seed_all(CONFIG["seed"])

e2e_model, E2E_MODEL_RUNS_ON_CPU = build_model(CONFIG, DEVICE, HAVE_MAMBA_SSM)
e2e_model = e2e_model.to(DEVICE)

if CODEC_DEVICE == "cpu" and DEVICE == "cuda" and not E2E_MODEL_RUNS_ON_CPU:
    raise RuntimeError("CPU codec cannot run this CUDA-only Mamba model.")

e2e_optimizer = torch.optim.AdamW(e2e_model.parameters(), lr=CONFIG["lr"], weight_decay=0.01)
e2e_criterion = torch.nn.CrossEntropyLoss()

e2e_tracker = EmissionsTracker(
    output_dir="./code_carbon/",
    output_file=e2e_csv,
    log_level="error",
)

sync_cuda()
e2e_tracker.start()
e2e_t0 = time.time()

train(
    e2e_model,
    train_loader,
    val_loader,
    test_loader,
    e2e_optimizer,
    e2e_criterion,
    device=DEVICE,
    name="baseline_boa_end_to_end",
    NUM_EPOCHS=CONFIG["epochs"],
    PRECISION=CONFIG["precision"],
    progress=True,
    vocab_size=CONFIG["vocab_size"],
)

e2e_boa = BOA(CODEC_DEVICE, "baseline_boa_end_to_end.boa", e2e_model)
e2e_boa.compress(DATA_PATH, chunks_count=chunks_count, progress=True)
e2e_restored = e2e_boa.decompress(progress=True)

sync_cuda()
e2e_emissions = e2e_tracker.stop()
e2e_runtime_s = time.time() - e2e_t0

assert e2e_restored == data, "end-to-end baseline BOA round-trip failed"

e2e_row = pd.read_csv(e2e_csv_path).iloc[-1]
e2e_energy_col = "energy_consumed" if "energy_consumed" in e2e_row else "energy_kWh"

e2e_boa_bytes = os.path.getsize("baseline_boa_end_to_end.boa")
e2e_model_bytes = state_size_bytes(e2e_model, "baseline_boa_end_to_end")
e2e_input_mb = len(data) / 1e6
e2e_saved_mb = (len(data) - e2e_boa_bytes) / 1e6

baseline_e2e_carbon = pd.DataFrame([{
    "method": "baseline_boa_end_to_end",
    "scope": "training + compression + decompression",
    "input_MB": e2e_input_mb,
    "actual_bits_per_byte": 8 * e2e_boa_bytes / len(data),
    "ratio_excl_model": len(data) / e2e_boa_bytes,
    "ratio_incl_model": len(data) / (e2e_boa_bytes + e2e_model_bytes),
    "runtime_s_codecarbon": float(e2e_row.get("duration", e2e_runtime_s)),
    "energy_kWh": float(e2e_row[e2e_energy_col]),
    "emissions_kgCO2eq": float(e2e_emissions if e2e_emissions is not None else e2e_row["emissions"]),
    "kgCO2eq_per_input_MB": float(e2e_row["emissions"]) / e2e_input_mb,
    "kgCO2eq_per_saved_MB": float(e2e_row["emissions"]) / e2e_saved_mb if e2e_saved_mb > 0 else float("nan"),
    "compressed_size_bytes": e2e_boa_bytes,
    "model_size_bytes": e2e_model_bytes,
    "roundtrip_ok": True,
}])

baseline_e2e_carbon.to_csv("baseline_boa_end_to_end_carbon_summary.csv", index=False)

display(baseline_e2e_carbon.style.format({
    "input_MB": "{:.3f}",
    "actual_bits_per_byte": "{:.4f}",
    "ratio_excl_model": "{:.3f}",
    "ratio_incl_model": "{:.3f}",
    "runtime_s_codecarbon": "{:.1f}",
    "energy_kWh": "{:.6e}",
    "emissions_kgCO2eq": "{:.6e}",
    "kgCO2eq_per_input_MB": "{:.6e}",
    "kgCO2eq_per_saved_MB": "{:.6e}",
}).hide(axis="index"))

print("saved: baseline_boa_end_to_end_carbon_summary.csv")
print("saved:", e2e_csv_path)

[model] mambav1 via mamba_ssm (fused CUDA)
[INFO] Using precision = fp32


Epoch 1 [fp32]:   0%|          | 0/798 [00:00<?, ?it/s]

  Epoch 1 throughput: 888,187 tok/s (0.85 MB/s), 44.9s
[Epoch 1] val bpp=2.3608 (ratio ~ 3.39x)  lr=5.00e-04


Epoch 2 [fp32]:   0%|          | 0/798 [00:00<?, ?it/s]

  Epoch 2 throughput: 856,061 tok/s (0.82 MB/s), 46.6s
[Epoch 2] val bpp=2.2705 (ratio ~ 3.52x)  lr=5.00e-04


...

Decompress (GPU streams x5000):   0%|          | 0.0/9995.1171875 [00:00<?, ?KB/s]

Decompress (GPU streams x5000):   0%|          | 0.0/9995.1171875 [00:00<?, ?KB/s]

Decompress (GPU streams x4375):   0%|          | 0.0/8745.7275390625 [00:00<?, ?KB/s]

method,scope,input_MB,actual_bits_per_byte,ratio_excl_model,ratio_incl_model,runtime_s_codecarbon,energy_kWh,emissions_kgCO2eq,kgCO2eq_per_input_MB,kgCO2eq_per_saved_MB,compressed_size_bytes,model_size_bytes,roundtrip_ok
baseline_boa_end_to_end,training + compression + decompression,49.920,2.1489,3.723,3.540,449.4,9.899081e-03,1.373496e-03,2.751394e-05,3.761902e-05,13409330,691885,True


saved: baseline_boa_end_to_end_carbon_summary.csv
saved: code_carbon/baseline_boa_end_to_end_emissions.csv
